In [ ]:
%load_ext autoreload
%autoreload 2
import h5py
import emcee
import dynesty
import corner
import pandas as pd
import numpy as np
import arviz as az
import json
import chainconsumer
from matplotlib.lines import Line2D
import sys
sys.path.insert(0, '/Users/padmavenkatraman/Documents/StrongLensing/fastTDC')
import tdc_sampler
from Utils.inference_utils import median_sigma_from_samples
from Utils.mcmc_utils import median_and_uncertainty, twoD_area, DE_fom
import Experiments.lsst_forecast.DataVectors.prep_data_vectors as prep_data_vectors

import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcdefaults()

Load in chains from Sherlock

In [ ]:
"""
'exp1_1':{
    'lcdm':{},
    'w0wa2':{},
},
'exp1_2':{
    'lcdm':{},
    'w0wa2':{},
},
"""
chains_dict = {'exp0_2':{'lCDM_seed1':{}, 'lCDM_seed2':{}, 'lCDM_seed3':{}, 
                         'lCDM_seed4':{}, 'lCDM_seed5':{}, 'lCDM_seed6':{}, 'lCDM_seed10':{}, 'lCDM_seed11':{}, 'lCDM_seed12':{},'lCDM_seed13':{}},
               'exp0_3':{'lCDM_seed1':{}, 'lCDM_seed2':{}, 'lCDM_seed4':{}},
               'exp0_4':{'lCDM_seed1':{}, 'lCDM_seed2':{}},
               'exp0_5':{'lCDM_seed1':{}, 'lCDM_seed2':{},'lCDM_seed3':{},'lCDM_seed4':{},'lCDM_seed5':{},'lCDM_seed6':{}, 'lCDM_seed7':{}},
               'exp0_6':{'lCDM_seed1':{}}}
for exp in chains_dict.keys():
    exp_dict = chains_dict[exp]
    for cosmo in exp_dict.keys():
        chain_path = 'Experiments/multiband_timedelays/InferenceRuns/'+exp+'/'+''+cosmo+'_backend.h5'
        print(chain_path)
        reader = emcee.backends.HDFBackend(chain_path, read_only=True)
        chains_dict[exp][cosmo]['chain'] = reader.get_chain()


### EXTERNAL: Load in chain from Shajib '24 ###

In [ ]:
# with h5py.File('Inferenceruns/Shajib_lensed_quasar_w0waCDM.h5','r') as h5:
#     print(h5['mcmc'].keys())
#     shajib24_accepted = h5['mcmc']['accepted'][:]
#     shajib24_chain = h5['mcmc']['chain'][:]

### EXTERNAL: Load in chain from TDCOSMO2025 ###

In [ ]:
# with h5py.File('../TDCOSMO2025/chains_export/Uw_0w_aCDM.h5','r') as h5:
#     tdcosmo2025_samples = h5['samples'][:]
#     tdcosmo2025_params = h5['parameters'][:]
#     print(tdcosmo2025_params)
#     tdcosmo2025_cosmo_only_chain = tdcosmo2025_samples[:,:4]

Let's check autocorrelation time

In [ ]:
exp = 'exp0_6'
cosmo = 'lCDM_seed1'
chain_path = 'Experiments/multiband_timedelays/InferenceRuns/'+exp+'/'+cosmo+'_backend.h5'
print(chain_path)
reader = emcee.backends.HDFBackend(chain_path, read_only=True)
reader.get_autocorr_time(discard=100)

In [ ]:
chain_path = 'Experiments/multiband_timedelays/InferenceRuns/exp0_6/lCDM_seed1_backend.h5'
print(chain_path)
reader = emcee.backends.HDFBackend(chain_path, read_only=True)
test_chain = reader.get_chain()

In [ ]:
test_chain.shape

In [ ]:
param_mcmc = ['$H_0$','$\Omega_M$','$\mu(\lambda_{int})$','$\sigma(\lambda_{int})$',
     r'$\mu(\beta_{ani})$',r'$\sigma(\beta_{ani})$',
     r'$\mu(\gamma_{lens})$',r'$\sigma(\gamma_{lens})$',
     r'$\mu(unc)$', r'$\sigma(unc)$']

In [ ]:
number_param = 3

plt.plot(test_chain[:, :,number_param], color='gray', alpha=0.3);

# plt.plot(test_chain[100:300, :, 0], color='black', alpha=0.3);
plt.ylabel(param_mcmc[number_param])
plt.xlabel('step')

In [ ]:
def plot_convergence_by_walker(samples_mcmc, param_mcmc, truths=None, verbose = False):
    n_params = samples_mcmc.shape[2]
    n_step = int(samples_mcmc.shape[1])
    chain = samples_mcmc
    mean_pos = np.zeros((n_params, n_step))
    median_pos = np.zeros((n_params, n_step))
    std_pos = np.zeros((n_params, n_step))
    q16_pos = np.zeros((n_params, n_step))
    q84_pos = np.zeros((n_params, n_step))
    # chain = np.empty((nwalker, nstep, ndim), dtype = np.double)
    for i in np.arange(n_params):
        for j in np.arange(n_step):
            mean_pos[i][j] = np.mean(chain[:, j, i])
            median_pos[i][j] = np.median(chain[:, j, i])
            std_pos[i][j] = np.std(chain[:, j, i])
            q16_pos[i][j] = np.percentile(chain[:, j, i], 16.)
            q84_pos[i][j] = np.percentile(chain[:, j, i], 84.)
    fig, ax = plt.subplots(n_params, sharex=True, figsize=(16, 2 * n_params))
    if n_params == 1: ax = [ax]
    last = n_step
    burnin = int((9.*n_step) / 10.) #get the final value on the last 10% on the chain
    for i in range(n_params):
        if truths is not None:
            ax[i].axhline(truths[i], c='b', lw=1)
        if verbose :
            print(param_mcmc[i], '{:.4f} +/- {:.4f}'.format(median_pos[i][last - 1], (q84_pos[i][last - 1] - q16_pos[i][last - 1]) / 2))
        ax[i].plot(median_pos[i][:last], c='g')
        ax[i].axhline(np.median(median_pos[i][burnin:last]), c='r', lw=1)
        ax[i].fill_between(np.arange(last), q84_pos[i][:last], q16_pos[i][:last], alpha=0.4)
        ax[i].set_ylabel(param_mcmc[i], fontsize=10)
        ax[i].set_xlim(0, last)
    return fig

#with h5py.File('DataVectors/gold/baseline_chain.h5','r') as h5:
#    test_chain = h5['mcmc_chain'][:]
#plot_convergence_by_walker(np.transpose(chains_dict['exp0_2']['w0wa_seed6_TdcosmoPrior']['chain'],axes=(1,0,2)),
# H0,Omega_M,mu_lambda_int,sigma_lambda_int,
        #   mu_beta_ani,sigma_beta_ani,mu_gamma,sigma_gamma
if 'lCDM' in cosmo:
    param_mcmc = ['$H_0$','$\Omega_M$','$\mu(\lambda_{int})$','$\sigma(\lambda_{int})$',
     r'$\mu(\beta_{ani})$',r'$\sigma(\beta_{ani})$',
     r'$\mu(\gamma_{lens})$',r'$\sigma(\gamma_{lens})$',
     r'$\mu(unc)$', r'$\sigma(unc)$']
elif 'w0wa' in cosmo:
    param_mcmc = ['$H_0$','$\Omega_M$','$w_0$','$w_a$',
     r'$\mu(\lambda_{int})$',r'$\sigma(\lambda_{int})$',
     r'$\mu(\beta_{ani})$',r'$\sigma(\beta_{ani})$',
     r'$\mu(\gamma_{lens})$',r'$\sigma(\gamma_{lens})$']
plot_convergence_by_walker(np.transpose(test_chain,axes=(1,0,2)),param_mcmc, truths=[70.,0.3,1, 0.1, 0, 0.1, 2.03, 0.2, 0.1, 0.001])


In [ ]:
the_chain = np.transpose(chains_dict['exp0_6']['lCDM_seed1']['chain'],axes=(1,0,2))
median_and_uncertainty(the_chain,-8000)

In [ ]:
the_chain = np.transpose(chains_dict['exp0_2']['lCDM_seed11']['chain'],axes=(1,0,2))
median_and_uncertainty(the_chain,-8000)

In [ ]:
# test twoD area code 
from scipy.stats import multivariate_normal

# test_chain = multivariate_normal.rvs(mean=[0.,0.],cov=np.diag([.5**2,.4**2]),size=(50,5000))
# test_chain.shape

# twoD_area(test_chain,[0,1],0,1000)

In [ ]:
categories =['Centered on truth','Joint','y', 'r', 'i', 'z', 'Sampled from dist\ncentered on truth' ]
chains = [chains_dict['exp0_5']['lCDM_seed6']['chain'][500:,:,:6].reshape(-1,6),
         chains_dict['exp0_2']['lCDM_seed11']['chain'][500:,:,:6].reshape(-1,6),
         chains_dict['exp0_5']['lCDM_seed2']['chain'][500:,:,:6].reshape(-1,6),
                     chains_dict['exp0_5']['lCDM_seed3']['chain'][500:,:,:6].reshape(-1,6),
                     chains_dict['exp0_5']['lCDM_seed4']['chain'][500:,:,:6].reshape(-1,6),
                     chains_dict['exp0_5']['lCDM_seed5']['chain'][500:,:,:6].reshape(-1,6),
                     chains_dict['exp0_5']['lCDM_seed7']['chain'][500:,:,:6].reshape(-1,6)]
# print each category's name, median and uncertainty for H0, omega_m, mu_lambda_int
for i, chain in enumerate(chains):
    median, uncertainty = np.median(chain, axis=0), np.std(chain, axis=0)
    print(categories[i], 'H0: {:.2f} +/- {:.2f}, Omega_m: {:.2f} +/- {:.2f}, mu_lambda_int: {:.2f} +/- {:.2f}'.format(
        median[0], uncertainty[0], median[1], uncertainty[1], median[2], uncertainty[2]))

In [ ]:
# use make_contour to plot the 2D contours for the parameters of interest.
from all_plotting_functions import make_contour
colors = {'u': '#1600ea', 'g': '#31de1f', 'r': '#b52626', 'i': '#370201', 'z': '#ba52ff', 'y': '#61a2b3'}
param_mcmc = np.array(param_mcmc)
burnin=2500
figure = make_contour(
    list_of_dists = [chains_dict['exp0_6']['lCDM_seed1']['chain'][500:,:,[0, 1, 2, 3, 4, 5, 8, 9]].reshape(-1,8)],
    labels=param_mcmc[[0, 1, 2, 3, 4, 5, 8, 9]], 
    categories = ['with_systematic_uncertainty'], 
    colors=['black'],
    range_for_bin=True, 
    show_correlation=False, 
    truths_list =[[70.,0.3,1, 0.1, 0, 0.1, 0.1, 0.001]] ,
    show_every_title=True,
    plot_lines=False)
# #370201

axes = np.array(figure.axes).reshape((8,8))
#h0, omega_m, mu_lambda_int, sigma_lambda_int, mu_beta_ani, sigma_beta_ani #### mu_gamma #### sigma_gamma
bounds = [[56,80],[0.16, 0.54],[0.92,1.1], [0.0,0.18],[-0.15,0.15],[0.0,0.2], [0, 10], [0, 0.01]]
for r in range(0,8):
        for c in range(0,r+1):
            if bounds is not None:
                axes[r,c].set_xlim(bounds[c])
                if r != c :
                    axes[r,c].set_ylim(bounds[r])


# figure.suptitle("LSST Imaging + 4MOST Kinematics", size=40)


In [ ]:
# use make_contour to plot the 2D contours for the parameters of interest.
from all_plotting_functions import make_contour
colors = {'u': '#1600ea', 'g': '#31de1f', 'r': '#b52626', 'i': '#370201', 'z': '#ba52ff', 'y': '#61a2b3'}

burnin=2500
figure = make_contour(
    list_of_dists = [
         
         chains_dict['exp0_2']['lCDM_seed11']['chain'][500:,:,:6].reshape(-1,6),
         chains_dict['exp0_5']['lCDM_seed2']['chain'][500:,:,:6].reshape(-1,6),
                     chains_dict['exp0_5']['lCDM_seed3']['chain'][500:,:,:6].reshape(-1,6),
                     chains_dict['exp0_5']['lCDM_seed4']['chain'][500:,:,:6].reshape(-1,6),
                     chains_dict['exp0_5']['lCDM_seed5']['chain'][500:,:,:6].reshape(-1,6),
                     chains_dict['exp0_5']['lCDM_seed6']['chain'][500:,:,:6].reshape(-1,6)],
    labels=param_mcmc[:6], 
    categories = ['Joint','y', 'r', 'i', 'z' ,'Centered on truth'], 
    colors=['gray', colors['y'], colors['r'], colors['i'], colors['z'],'orange'],
    range_for_bin=True, 
    show_correlation=False, 
    truths_list =[[70.,0.3,1, 0.1, 0, 0.1],[70.,0.3,1, 0.1, 0, 0.1],
                  [70.,0.3,1, 0.1, 0, 0.1],
                  [70.,0.3,1, 0.1, 0, 0.1],
                  [70.,0.3,1, 0.1, 0, 0.1],
                  [70.,0.3,1, 0.1, 0, 0.1]] ,
    show_every_title=True,
    plot_lines=False)
# #370201

axes = np.array(figure.axes).reshape((6,6))
#h0, omega_m, mu_lambda_int, sigma_lambda_int, mu_beta_ani, sigma_beta_ani #### mu_gamma #### sigma_gamma
bounds = [[56,80],[0.16, 0.54],[0.92,1.1], [0.0,0.18],[-0.15,0.15],[0.0,0.2]]
for r in range(0,6):
        for c in range(0,r+1):
            if bounds is not None:
                axes[r,c].set_xlim(bounds[c])
                if r != c :
                    axes[r,c].set_ylim(bounds[r])


# figure.suptitle("LSST Imaging + 4MOST Kinematics", size=40)


In [ ]:
# use make_contour to plot the 2D contours for the parameters of interest.
from all_plotting_functions import make_contour
colors = {'u': '#1600ea', 'g': '#31de1f', 'r': '#b52626', 'i': '#370201', 'z': '#ba52ff', 'y': '#61a2b3'}

burnin=2500
figure = make_contour(
    list_of_dists = [
         
         chains_dict['exp0_2']['lCDM_seed11']['chain'][500:,:,:6].reshape(-1,6),
         chains_dict['exp0_5']['lCDM_seed2']['chain'][500:,:,:6].reshape(-1,6),
                     chains_dict['exp0_5']['lCDM_seed3']['chain'][500:,:,:6].reshape(-1,6),
                     chains_dict['exp0_5']['lCDM_seed4']['chain'][500:,:,:6].reshape(-1,6),
                     chains_dict['exp0_5']['lCDM_seed5']['chain'][500:,:,:6].reshape(-1,6)],
    labels=param_mcmc[:6], 
    categories = ['Joint','y', 'r', 'i', 'z' ], 
    colors=['gray', colors['y'], colors['r'], colors['i'], colors['z']],
    range_for_bin=True, 
    show_correlation=False, 
    truths_list =[[70.,0.3,1, 0.1, 0, 0.1],[70.,0.3,1, 0.1, 0, 0.1],
                  [70.,0.3,1, 0.1, 0, 0.1],
                  [70.,0.3,1, 0.1, 0, 0.1],
                  [70.,0.3,1, 0.1, 0, 0.1],
                  ] ,
    show_every_title=True,
    plot_lines=False)
# #370201

axes = np.array(figure.axes).reshape((6,6))
#h0, omega_m, mu_lambda_int, sigma_lambda_int, mu_beta_ani, sigma_beta_ani #### mu_gamma #### sigma_gamma
bounds = [[56,80],[0.16, 0.54],[0.92,1.1], [0.0,0.18],[-0.15,0.15],[0.0,0.2]]
for r in range(0,6):
        for c in range(0,r+1):
            if bounds is not None:
                axes[r,c].set_xlim(bounds[c])
                if r != c :
                    axes[r,c].set_ylim(bounds[r])


# figure.suptitle("LSST Imaging + 4MOST Kinematics", size=40)


In [ ]:
# use make_contour to plot the 2D contours for the parameters of interest.
from all_plotting_functions import make_contour
colors = {'u': '#1600ea', 'g': '#31de1f', 'r': '#b52626', 'i': '#370201', 'z': '#ba52ff', 'y': '#61a2b3'}

burnin=2500
figure = make_contour(
    list_of_dists = [
         
         chains_dict['exp0_2']['lCDM_seed11']['chain'][500:,:,:6].reshape(-1,6),
         chains_dict['exp0_5']['lCDM_seed2']['chain'][500:,:,:6].reshape(-1,6),
                     chains_dict['exp0_5']['lCDM_seed3']['chain'][500:,:,:6].reshape(-1,6),
                     chains_dict['exp0_5']['lCDM_seed4']['chain'][500:,:,:6].reshape(-1,6),
                     chains_dict['exp0_5']['lCDM_seed5']['chain'][500:,:,:6].reshape(-1,6),
                     chains_dict['exp0_5']['lCDM_seed6']['chain'][500:,:,:6].reshape(-1,6)],
    labels=param_mcmc[:6], 
    categories = ['Joint','y', 'r', 'i', 'z' ], 
    colors=['gray', colors['y'], colors['r'], colors['i'], colors['z'],'orange'],
    range_for_bin=True, 
    show_correlation=False, 
    truths_list =[[70.,0.3,1, 0.1, 0, 0.1],[70.,0.3,1, 0.1, 0, 0.1],
                  [70.,0.3,1, 0.1, 0, 0.1],
                  [70.,0.3,1, 0.1, 0, 0.1],
                  [70.,0.3,1, 0.1, 0, 0.1],
                  [70.,0.3,1, 0.1, 0, 0.1]] ,
    show_every_title=True,
    plot_lines=False)
# #370201

axes = np.array(figure.axes).reshape((6,6))
#h0, omega_m, mu_lambda_int, sigma_lambda_int, mu_beta_ani, sigma_beta_ani #### mu_gamma #### sigma_gamma
bounds = [[56,80],[0.16, 0.54],[0.92,1.1], [0.0,0.18],[-0.15,0.15],[0.0,0.2]]
for r in range(0,6):
        for c in range(0,r+1):
            if bounds is not None:
                axes[r,c].set_xlim(bounds[c])
                if r != c :
                    axes[r,c].set_ylim(bounds[r])


# figure.suptitle("LSST Imaging + 4MOST Kinematics", size=40)


In [ ]:
# use make_contour to plot the 2D contours for the parameters of interest.
from all_plotting_functions import make_contour
colors = {'u': '#1600ea', 'g': '#31de1f', 'r': '#b52626', 'i': '#370201', 'z': '#ba52ff', 'y': '#61a2b3'}

burnin=2500
figure = make_contour(
    list_of_dists = [
         
         chains_dict['exp0_2']['lCDM_seed11']['chain'][500:,:,:6].reshape(-1,6),
        #  chains_dict['exp0_5']['lCDM_seed2']['chain'][500:,:,:6].reshape(-1,6),
        #              chains_dict['exp0_5']['lCDM_seed3']['chain'][500:,:,:6].reshape(-1,6),
        #              chains_dict['exp0_5']['lCDM_seed4']['chain'][500:,:,:6].reshape(-1,6),
        #              chains_dict['exp0_5']['lCDM_seed5']['chain'][500:,:,:6].reshape(-1,6),
                     chains_dict['exp0_5']['lCDM_seed6']['chain'][500:,:,:6].reshape(-1,6),
                     chains_dict['exp0_5']['lCDM_seed7']['chain'][500:,:,:6].reshape(-1,6),],
    labels=param_mcmc[:6], 
    categories = ['Joint','Centered on truth (COT)', 'Sampled from dist (COT)'], 
    colors=['gray', 'orange', 'green'],
    range_for_bin=True, 
    show_correlation=False, 
    truths_list =[[70.,0.3,1, 0.1, 0, 0.1],[70.,0.3,1, 0.1, 0, 0.1],
                  [70.,0.3,1, 0.1, 0, 0.1],
                 ] ,
    show_every_title=True,
    plot_lines=False)
# #370201

axes = np.array(figure.axes).reshape((6,6))
#h0, omega_m, mu_lambda_int, sigma_lambda_int, mu_beta_ani, sigma_beta_ani #### mu_gamma #### sigma_gamma
bounds = [[56,80],[0.16, 0.54],[0.92,1.1], [0.0,0.18],[-0.15,0.15],[0.0,0.2]]
for r in range(0,6):
        for c in range(0,r+1):
            if bounds is not None:
                axes[r,c].set_xlim(bounds[c])
                if r != c :
                    axes[r,c].set_ylim(bounds[r])


# figure.suptitle("LSST Imaging + 4MOST Kinematics", size=40)


In [ ]:
# use make_contour to plot the 2D contours for the parameters of interest.
from all_plotting_functions import make_contour
burnin=2500
figure = make_contour(
    list_of_dists = [chains_dict['exp0_2']['lCDM_seed11']['chain'][500:,:,:6].reshape(-1,6),
                     chains_dict['exp0_5']['lCDM_seed2']['chain'][500:,:,:6].reshape(-1,6),
                     chains_dict['exp0_2']['lCDM_seed13']['chain'][500:,:,:6].reshape(-1,6)],
    labels=param_mcmc[:6], 
    categories = ['LSST_td_joint', "LSST_td_y_including_high_uncertainty", "LSST_centered_on_truth"], 
    colors=['gray', 'turquoise', 'red'], 
    range_for_bin=True, 
    show_correlation=True, 
    truths_list =[[70.,0.3,1, 0.1, 0, 0.1],
                  
                  [70.,0.3,1, 0.1, 0, 0.1],
                  [70.,0.3,1, 0.1, 0, 0.1]] ,
    show_every_title=True)
# #370201

axes = np.array(figure.axes).reshape((6,6))
#h0, omega_m, mu_lambda_int, sigma_lambda_int, mu_beta_ani, sigma_beta_ani #### mu_gamma #### sigma_gamma
bounds = [[56,80],[0.16, 0.54],[0.92,1.1], [0.0,0.18],[-0.15,0.15],[0.0,0.2]]
for r in range(0,6):
        for c in range(0,r+1):
            if bounds is not None:
                axes[r,c].set_xlim(bounds[c])
                if r != c :
                    axes[r,c].set_ylim(bounds[r])


figure.suptitle("LSST Imaging + 4MOST Kinematics", size=40)


In [ ]:
(64-69)/69

In [ ]:
(0.96-0.99)/0.99

In [ ]:
# [70.,0.3,1, 0.1, 0, 0.1, 2.03, 0.2],
# 

In [ ]:
# use make_contour to plot the 2D contours for the parameters of interest.
from all_plotting_functions import make_contour
burnin=2500
figure = make_contour(
    list_of_dists = [chains_dict['exp0_4']['lCDM_seed2']['chain'][burnin:,:,:].reshape(-1,8),
                     
                     chains_dict['exp0_2']['lCDM_seed6']['chain'][burnin:,:,:].reshape(-1,8),] ,
    labels=param_mcmc, 
    categories = ['LSST_td_i_band','LSST_td_joint'], 
    colors=['darkred','black'], 
    range_for_bin=True, 
    show_correlation=True, 
    truths_list =[[70.,0.3,1, 0.1, 0, 0.1, 2.03, 0.2],
                  
                  [70.,0.3,1, 0.1, 0, 0.1, 2.03, 0.2]] ,
    show_every_title=True)
# 

axes = np.array(figure.axes).reshape((8,8))
#h0, omega_m, mu_lambda_int, sigma_lambda_int, mu_beta_ani, sigma_beta_ani #### mu_gamma #### sigma_gamma
bounds = [[56,80],[0.16, 0.54],[0.92,1.1], [0.0,0.18],[-0.15,0.15],[0.0,0.2],[1.5,2.5],[0.0,0.2]]
for r in range(0,8):
        for c in range(0,r+1):
            if bounds is not None:
                axes[r,c].set_xlim(bounds[c])
                if r != c :
                    axes[r,c].set_ylim(bounds[r])


figure.suptitle("LSST Imaging + 4MOST Kinematics", size=40)


In [ ]:
# use make_contour to plot the 2D contours for the parameters of interest.
from all_plotting_functions import make_contour
burnin=2500
figure = make_contour(
    list_of_dists = [chains_dict['exp0_3']['lCDM_seed1']['chain'][burnin:,:,:].reshape(-1,8),
                     chains_dict['exp0_3']['lCDM_seed2']['chain'][burnin:,:,:].reshape(-1,8),
                     chains_dict['exp0_3']['lCDM_seed4']['chain'][200:,:,:].reshape(-1,8)] ,
    labels=param_mcmc, 
    categories = ['5 day', 'LSST_td', 'LSST_td_centered_on_truth'], 
    colors=['red', 'green','blue'], 
    range_for_bin=False, 
    show_correlation=True, 
    truths_list =[[70.,0.3,1, 0.1, 0, 0.1, 2.03, 0.2],
                  [70.,0.3,1, 0.1, 0, 0.1, 2.03, 0.2],
                  [70.,0.3,1, 0.1, 0, 0.1, 2.03, 0.2],
                  ] )


axes = np.array(figure.axes).reshape((8,8))
#h0, omega_m, mu_lambda_int, sigma_lambda_int, mu_beta_ani, sigma_beta_ani #### mu_gamma #### sigma_gamma
bounds = [[56,80],[0.16, 0.54],[0.92,1.1], [0.0,0.18],[-0.15,0.15],[0.0,0.2],[1.6,2.4],[0.0,0.2]]
for r in range(0,8):
        for c in range(0,r+1):
            if bounds is not None:
                axes[r,c].set_xlim(bounds[c])
                if r != c :
                    axes[r,c].set_ylim(bounds[r])

figure.suptitle("HST imaging + MUSE kinematics", size=40)

#### understand bias

In [ ]:
# use corner to plot the 2D contours of the parameters
figure = corner.corner(
    np.transpose(chains_dict['exp0_2']['lCDM_seed1']['chain'],axes=(1,0,2))[-10000:,:,:].reshape(-1,8),
    labels=param_mcmc,
    show_titles=True,
    title_fmt='.3f',
    title_kwargs={"fontsize": 12},
    quantiles=[0.16,0.5,0.84],
    label_kwargs={"fontsize": 12},
    plot_datapoints=False,
    plot_density=True,
    plot_contours=True,
    smooth=1.0)


# axes = np.array(figure.axes).reshape((8,8))
# bounds = [[60,100],[-0.16, 0.48],[0.92,0.11], [0.0,0.5],[-0.1,0.1],[0.0,0.5],[0.9,1.1],[0.0,0.5]]
# for r in range(0,3):
#         for c in range(0,r+1):
#             if bounds is not None:
#                 axes[r,c].set_xlim(bounds[c])
#                 if r != c :
#                     axes[r,c].set_ylim(bounds[r])


# axes[0,2].legend(custom_lines,custom_labels,frameon=False,fontsize=15)

In [ ]:
import matplotlib
metadata_gold = pd.read_csv('DataVectors/gold/truth_metadata.csv')
gamma_lens_means = []
gamma_lens_stddevs = []
#for catalog_idx_list in [
#                         chains_dict['exp0_2']['w0wa_seed1']['catalog_idx']]:
#    truth_gammas = metadata_gold['main_deflector_parameters_gamma'][
#        np.isin(metadata_gold['catalog_idx'],catalog_idx_list)]
#    gamma_lens_means.append(np.mean(truth_gammas))
#s    gamma_lens_stddevs.append(np.std(truth_gammas))


exp_chains = [
    np.asarray([tdcosmo2025_cosmo_only_chain]),
    #np.transpose(chains_dict['exp1_3']['w0wa_seed6_TdcosmoPrior']['chain'],axes=(1,0,2)),
    #np.transpose(chains_dict['exp1_2']['w0wa_seed6_TdcosmoPrior']['chain'],axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed6_TdcosmoPrior']['chain'],axes=(1,0,2))]
exp_names = ['TDCOSMO 2025 \n 8 Lenses (Real)',
            #'50 IFU Lenses (Simulated)',
            #'+150 Gold Lenses (Simulated)',
            'DESC Forecast (Erickson '+chr(39)+'25) \n 800 Lenses (Simulated)']
             #'Exp 0.2: Baseline, Seed 1, Steps 32k-44k']

num_chains = len(exp_chains)
burnin = [100000,10000,20000,20000]
colors = ['black',"#00A9A9"] # "#BAB4D7","#629FCE"#goldenrod',
truth_colors = ["#000000"] * num_chains

custom_lines = []
custom_labels = []

for i,exp_chain in enumerate(exp_chains):

    num_params = exp_chain.shape[2]

    my_color = colors[i]
    
    print(exp_names[i])
    median_and_uncertainty(exp_chain,burnin[i])
    zp,fom = DE_fom(exp_chain,burnin[i])
     
    if i ==0:

        figure = corner.corner(exp_chain[:,burnin[i]:,[0,2,3]].reshape((-1,3)),plot_datapoints=False,
            color=my_color,levels=[0.68,0.95],fill_contours=False,plot_density=False,
            labels= ['$H_0$','$w_0$','$w_a$'],
            dpi=300,hist_kwargs={'density':True},
            fig=None,label_kwargs={'fontsize':20},smooth=2)

    else:

        corner.corner(exp_chain[:,burnin[i]:,[0,2,3]].reshape((-1,3)),plot_datapoints=False,
            color=my_color,levels=[0.68,0.95],fill_contours=True,
            labels= ['$H_0$','$w_0$','$w_a$'],
            dpi=300,hist_kwargs={'density':True},
            fig=figure,label_kwargs={'fontsize':40},smooth=2)
        
    custom_lines.append(Line2D([0], [0], color=my_color, lw=4))

    # calculate h0 constraint
    h0, h0_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,0].reshape((-1,1)),weights=None)
    OmegaM, OmegaM_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,1].reshape((-1,1)),weights=None)
    w0, w0_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,2].reshape((-1,1)),weights=None)
    wa, wa_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,3].reshape((-1,1)),weights=None)
    # construct label
    custom_labels.append(exp_names[i])
        #+
        #':\n $H_0$=%.1f$\pm$%.1f \n $\Omega_M$=%.2f$\pm$%.2f \n DE FOM = %.1f'%(
        #np.round(h0,decimals=1), np.round(h0_sigma,decimals=1), 
        #np.round(OmegaM,decimals=2), np.round(OmegaM_sigma,decimals=2), 
        #np.round(fom,decimals=1)))


axes = np.array(figure.axes).reshape((3, 3))
bounds = [[60,100],[-1.5,0.5],[-5,4.8]]
for r in range(0,3):
        for c in range(0,r+1):
            if bounds is not None:
                axes[r,c].set_xlim(bounds[c])
                if r != c :
                    axes[r,c].set_ylim(bounds[r])


axes[0,2].legend(custom_lines,custom_labels,frameon=False,fontsize=15)
#plt.tight_layout()
plt.savefig('/Users/smericks/Desktop/lsst_isolated.pdf',bbox_inches='tight')

In [ ]:
chains_dict['exp1_2']['w0wa_seed6']['chain'].shape

In [ ]:
import matplotlib
metadata_gold = pd.read_csv('DataVectors/gold/truth_metadata.csv')
gamma_lens_means = []
gamma_lens_stddevs = []
#for catalog_idx_list in [
#                         chains_dict['exp0_2']['w0wa_seed1']['catalog_idx']]:
#    truth_gammas = metadata_gold['main_deflector_parameters_gamma'][
#        np.isin(metadata_gold['catalog_idx'],catalog_idx_list)]
#    gamma_lens_means.append(np.mean(truth_gammas))
#s    gamma_lens_stddevs.append(np.std(truth_gammas))


exp_chains = [
    np.transpose(chains_dict['exp1_3']['w0wa_seed6']['chain'],axes=(1,0,2)),
    np.transpose(chains_dict['exp1_2']['w0wa_seed6']['chain'],axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed6_LONG']['chain'][:70000],axes=(1,0,2))]
exp_names = ['50 IFU Lenses',
            '+150 Gold Lenses',
             '+600 LSST Lenses']
             #'Exp 0.2: Baseline, Seed 1, Steps 32k-44k']

num_chains = len(exp_chains)
burnin = [10000,10000,20000]
cmap = plt.get_cmap('ocean')
colors = ["#BAB4D7","#629FCE","#00A9A9"] #goldenrod',
truth_colors = ["#000000"] * num_chains

custom_lines = []
custom_labels = []

for i,exp_chain in enumerate(exp_chains):

    num_params = exp_chain.shape[2]

    my_color = colors[i]
    
    print(exp_names[i])
    median_and_uncertainty(exp_chain,burnin[i])
    zp,fom = DE_fom(exp_chain,burnin[i])
     
    if i ==0:

        figure = corner.corner(exp_chain[:,burnin[i]:,:-2].reshape((-1,exp_chain.shape[2]-2)),plot_datapoints=False,
            color=my_color,levels=[0.68,0.95],fill_contours=True,
            labels= ['$H_0$','$\Omega_m$','$w_0$','$w_a$',
                r'$\mu(\lambda_{int})$',r'$\sigma(\lambda_{int})$',
                r'$\mu(\beta_{ani})$',r'$\sigma(\beta_{ani})$'],
            dpi=300,truths=[70.,0.3,-1.0,0.,
                1.,0.1,0.,0.1],truth_color=truth_colors[i],
            fig=None,label_kwargs={'fontsize':40},smooth=2)

    else:

        corner.corner(exp_chain[:,burnin[i]:,:-2].reshape((-1,exp_chain.shape[2]-2)),plot_datapoints=False,
            color=my_color,levels=[0.68,0.95],fill_contours=True,
            labels=['$H_0$','$\Omega_m$','$w_0$','$w_a$',
                r'$\mu(\lambda_{int})$',r'$\sigma(\lambda_{int})$',
                r'$\mu(\beta_{ani})$',r'$\sigma(\beta_{ani})$'],
            dpi=300,truths=[70.,0.3,-1.0,0.,
                1.,0.1,0.,0.1],truth_color=truth_colors[i],
            fig=figure,label_kwargs={'fontsize':40},smooth=2)
        
    custom_lines.append(Line2D([0], [0], color=my_color, lw=10))

    # calculate h0 constraint
    h0, h0_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,0].reshape((-1,1)),weights=None)
    OmegaM, OmegaM_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,1].reshape((-1,1)),weights=None)
    w0, w0_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,2].reshape((-1,1)),weights=None)
    wa, wa_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,3].reshape((-1,1)),weights=None)
    # construct label
    custom_labels.append(exp_names[i]+
        ':\n $H_0$=%.1f$\pm$%.1f \n $\Omega_m$=%.2f$\pm$%.2f \n DE FOM = %.1f'%(
        np.round(h0,decimals=1), np.round(h0_sigma,decimals=1), 
        np.round(OmegaM,decimals=2), np.round(OmegaM_sigma,decimals=2), 
        np.round(fom,decimals=1)))

"""
axes = np.array(figure.axes).reshape((3, 3))
bounds = [[63,77],[1.91,2.095],[0.0,0.2]]
for r in range(0,3):
        for c in range(0,r+1):
            if bounds is not None:
                axes[r,c].set_xlim(bounds[c])
                if r != c :
                    axes[r,c].set_ylim(bounds[r])

axes = np.array(figure.axes).reshape((3, 3))
"""

axes = np.array(figure.axes).reshape((num_params-2, num_params-2))
axes[0,num_params-3].legend(custom_lines,custom_labels,frameon=False,fontsize=25)
plt.tight_layout()
plt.savefig('/Users/smericks/Desktop/gold_vs_silver.pdf',bbox_inches='tight')

In [ ]:
exp_chains = [
    np.transpose(shajib24_chain,axes=(1,0,2))[:,:,:4],
    np.transpose(chains_dict['exp0_2']['w0wa_seed6_LONG']['chain'],axes=(1,0,2))[:,:70000,:4],
    np.transpose(chains_dict['exp5_1']['w0wa_seed6_LONG']['chain'],axes=(1,0,2))[:,:70000,:4]]
burnin = [10000,20000,20000]

exp_names = ['Shajib 2025: 236 Lenses',
            #'Exp 1.2: Gold-Only, Seed 0',
             'Exp 0.1: Baseline, 800 Lenses',
             'Exp 3.1: Extra LTM, 800 Lenses']
             #'Exp 0.2: Baseline, Seed 1, Steps 32k-44k']
colors = ['sandybrown','darkgrey','cornflowerblue'] #goldenrod',
truth_colors = ["#000000"] * len(exp_names)

custom_lines = []
custom_labels = []

for i,exp_chain in enumerate(exp_chains):

    num_params = exp_chain.shape[2]

    my_color = colors[i]
    
    print(exp_names[i])
    #median_and_uncertainty(exp_chain,burnin[i])
    print(exp_names[i])
    zp,fom = DE_fom(exp_chain,burnin[i])
     
    if i ==0:

        figure = corner.corner(exp_chain[:,burnin[i]:].reshape((-1,exp_chain.shape[2])),plot_datapoints=False,
            color=my_color,levels=[0.68,0.95],fill_contours=True,
            labels= ['$H_0$','$\Omega_m$','$w_0$','$w_a$'],
            dpi=300,truths=[70.,0.3,-1.0,0.],truth_color=truth_colors[i],
            fig=None,label_kwargs={'fontsize':24},hist_kwargs={'density':True},
            smooth=2)

    else:

        corner.corner(exp_chain[:,burnin[i]:].reshape((-1,exp_chain.shape[2])),plot_datapoints=False,
            color=my_color,levels=[0.68,0.95],fill_contours=True,
            labels=['$H_0$','$\Omega_m$','$w_0$','$w_a$'],
            dpi=300,truths=[70.,0.3,-1.0,0.],truth_color=truth_colors[i],
            fig=figure,label_kwargs={'fontsize':24},hist_kwargs={'density':True},
            smooth=2)
        
    custom_lines.append(Line2D([0], [0], color=my_color, lw=4))

    # calculate h0 constraint
    h0, h0_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,0].reshape((-1,1)),weights=None)
    OmegaM, OmegaM_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,1].reshape((-1,1)),weights=None)
    w0, w0_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,2].reshape((-1,1)),weights=None)
    wa, wa_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,3].reshape((-1,1)),weights=None)
    # construct label
    custom_labels.append(exp_names[i]+
        ':\n $H_0$=%.1f$\pm$%.1f \n $\Omega_m$=%.2f$\pm$%.2f \n DE FOM = %.1f'%(
        h0, h0_sigma, OmegaM, OmegaM_sigma, fom))


axes = np.array(figure.axes).reshape((4, 4))
bounds = [[62,78],[0.05,0.5],[-2,0.],[-2,2]]
for r in range(0,4):
        for c in range(0,r+1):
            if bounds is not None:
                axes[r,c].set_xlim(bounds[c])
                if r != c :
                    axes[r,c].set_ylim(bounds[r])

#axes = np.array(figure.axes).reshape((3, 3))


axes = np.array(figure.axes).reshape((4, 4))
axes[0,3].legend(custom_lines,custom_labels,frameon=False,fontsize=14)
plt.savefig('/Users/smericks/Desktop/comp_to_shajib24.pdf')

In [ ]:

exp_chains = [
    np.transpose(chains_dict['exp1_3']['w0wa_seed6']['chain'],axes=(1,0,2)),
    np.transpose(chains_dict['exp1_2']['w0wa_seed6']['chain'],axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed6_LONG']['chain'][:70000],axes=(1,0,2))]
exp_names = ['50 IFU Lenses',
            '+150 Gold Lenses',
             '+600 LSST Lenses']
             #'Exp 0.2: Baseline, Seed 1, Steps 32k-44k']

num_chains = len(exp_chains)
burnin = [10000,10000,20000]
cmap = plt.get_cmap('ocean')
colors = ["#BAB4D7","#629FCE","#00A9A9"] #goldenrod',

In [ ]:
exp_chains = [
    np.transpose(chains_dict['exp1_3']['w0wa_seed6']['chain'],axes=(1,0,2))[:,:70000,:4],
   # np.transpose(chains_dict['exp1_2']['w0wa_seed6']['chain'],axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed6_LONG']['chain'][:70000],axes=(1,0,2))[:,:70000,:4]]
exp_names = ['50 IFU Lenses',
             '+750 LSST Lenses']

num_chains = len(exp_chains)
burnin = [10000,10000,20000]
colors = ["#BAB4D7","#00A9A9"] #goldenrod', # "#629FCE"
truth_colors = ["#000000"] * num_chains

custom_lines = []
custom_labels = []

for i,exp_chain in enumerate(exp_chains):

    num_params = exp_chain.shape[2]

    my_color = colors[i]
    
    print(exp_names[i])
    #median_and_uncertainty(exp_chain,burnin[i])
    print(exp_names[i])
    zp,fom = DE_fom(exp_chain,burnin[i])
     
    if i ==0:

        figure = corner.corner(exp_chain[:,burnin[i]:,[0,2,3]].reshape((-1,3)),plot_datapoints=False,
            color=my_color,levels=[0.68,0.95],fill_contours=True,
            labels= ['$H_0$','$w_0$','$w_a$'],
            dpi=300,truths=[70.,-1.0,0.],truth_color=truth_colors[i],
            fig=None,label_kwargs={'fontsize':24},hist_kwargs={'density':True},
            smooth=2)

    else:

        corner.corner(exp_chain[:,burnin[i]:,[0,2,3]].reshape((-1,3)),plot_datapoints=False,
            color=my_color,levels=[0.68,0.95],fill_contours=True,
            labels=['$H_0$','$w_0$','$w_a$'],
            dpi=300,truths=[70.,-1.0,0.],truth_color=truth_colors[i],
            fig=figure,label_kwargs={'fontsize':24},hist_kwargs={'density':True},
            smooth=2)
        
    custom_lines.append(Line2D([0], [0], color=my_color, lw=4))

    # calculate h0 constraint
    h0, h0_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,0].reshape((-1,1)),weights=None)
    OmegaM, OmegaM_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,1].reshape((-1,1)),weights=None)
    w0, w0_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,2].reshape((-1,1)),weights=None)
    wa, wa_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,3].reshape((-1,1)),weights=None)
    # construct label
    custom_labels.append(exp_names[i]+
        ':\n $H_0$=%.1f$\pm$%.1f \n DE FOM = %.1f'%(
        h0, h0_sigma, fom))


axes = np.array(figure.axes).reshape((3, 3))
bounds = [[62,78],[-2,0.],[-2,2]]
for r in range(0,3):
        for c in range(0,r+1):
            if bounds is not None:
                axes[r,c].set_xlim(bounds[c])
                if r != c :
                    axes[r,c].set_ylim(bounds[r])

#axes = np.array(figure.axes).reshape((3, 3))


axes[0,2].legend(custom_lines,custom_labels,frameon=False,fontsize=17)
plt.savefig('/Users/smericks/Desktop/isolated_contours.pdf')

In [ ]:
with h5py.File('Inferenceruns/Shajib_lensed_quasar_w0waCDM.h5','r') as h5:
    print(h5['mcmc'].keys())
    shajib24_accepted = h5['mcmc']['accepted'][:]
    shajib24_chain = h5['mcmc']['chain'][:]

print('chain shape: ', shajib24_chain.shape)


exp_chain = np.transpose(shajib24_chain,axes=(1,0,2))[:,:,:4]
burnin=1000

print(exp_chain.shape)
test_chain = exp_chain[:,burnin:].reshape((-1,exp_chain.shape[2]))
#print(test_chain.shape)
consumer = chainconsumer.ChainConsumer().add_chain(test_chain)
consumer.plotter.plot()

In [ ]:
test_chain = np.transpose(chains_dict['exp0_2']['w0wa_seed6']['chain'],axes=(1,0,2))
burnin = 20000

chains = [
    np.transpose(chains_dict['exp0_2']['w0wa_seed6_LONG']['chain'][:70000],axes=(1,0,2)),
    np.transpose(chains_dict['exp2_1']['w0wa_seed6_LONG']['chain'][:70000],axes=(1,0,2)),
    np.transpose(chains_dict['exp2_2']['w0wa_seed6_LONG']['chain'][:70000],axes=(1,0,2)),
    np.transpose(chains_dict['exp4_1']['w0wa_seed6_LONG']['chain'][:70000],axes=(1,0,2)),
    np.transpose(chains_dict['exp4_2']['w0wa_seed6_LONG']['chain'][:70000],axes=(1,0,2)),
    np.transpose(chains_dict['exp4_3']['w0wa_seed6_LONG']['chain'][:70000],axes=(1,0,2)),
    np.transpose(chains_dict['exp5_1']['w0wa_seed6_LONG']['chain'][:70000],axes=(1,0,2)),
    np.transpose(chains_dict['exp5_2']['w0wa_seed6_LONG']['chain'][:70000],axes=(1,0,2)),
    np.transpose(chains_dict['exp5_3']['w0wa_seed6_LONG']['chain'][:70000],axes=(1,0,2)),
    np.transpose(chains_dict['exp5_4']['w0wa_seed6_LONG']['chain'][:70000],axes=(1,0,2))
]

labels = ['exp0_2','exp2_1','exp2_2','exp4_1','exp4_2','exp4_3','exp5_1','exp5_2','exp5_3','exp5_4']

for j,curr_chain in enumerate(chains):
    print(labels[j])
    median_and_uncertainty(curr_chain,burnin)
    DE_fom(curr_chain,burnin)

I need a H0,w0,wa only plot

In [ ]:
import matplotlib

metadata_gold = pd.read_csv('DataVectors/gold/truth_metadata.csv')

baseline_chain = np.transpose(chains_dict['exp0_2']['w0wa_seed6_LONG']['chain'][:70000], axes=(1,0,2))
baseline_label =  'Exp 0.2: Baseline, Seed 1'
exp_chains = [
    np.transpose(chains_dict['exp2_1']['w0wa_seed6_LONG']['chain'][:70000], axes=(1,0,2)),
    np.transpose(chains_dict['exp2_2']['w0wa_seed6_LONG']['chain'][:70000], axes=(1,0,2)),
    np.transpose(chains_dict['exp5_1']['w0wa_seed6_LONG']['chain'][:70000], axes=(1,0,2)),
    np.transpose(chains_dict['exp5_4']['w0wa_seed6_LONG']['chain'][:70000], axes=(1,0,2))
]
exp_names = [
    'Exp 1.1: Extra IFU Kin., Seed 5',
    'Exp 1.2: Extra Aperture Kin., Seed 5',
    'Exp 3.1: Extra Time-Delay Monitoring, Seed 5',
    'Exp 3.4: $\sigma(\Delta t)_{LSST}$ = 2 days, Seed 5'
]
num_chains = len(exp_chains)
burnin = [20000] * (num_chains+1)


exp_labels = [
    'Experiment 1.1', 'Experiment 1.2', 'Experiment 3.1', 'Experiment 3.4'
]

# Choose a color map for the contours
cmap = plt.get_cmap('cividis')
colors = ['darkgrey','seagreen','yellowgreen','cornflowerblue','blueviolet']
truth_colors = ["#000000"] * num_chains
custom_labels = []
custom_lines = []
custom_labels = []

for j,curr_chain in enumerate(exp_chains):
     
    print(exp_names[j]) 
    #median_and_uncertainty(curr_chain,burnin[j+1])
    zp,fom = DE_fom(curr_chain,burnin[i])

    figure = corner.corner(baseline_chain[:,burnin[0]:,[0,1,2,3]].reshape((-1,4)),plot_datapoints=False,
                color=colors[0],levels=[0.68,0.95],fill_contours=True,
                labels= ['$H_0$','$\Omega_m$','$w_0$','$w_a$'],
                dpi=300,truths=[70.,0.3,-1.0,0.],truth_color='black',
                fig=None,label_kwargs={'fontsize':22},smooth=2.)

    corner.corner(curr_chain[:,burnin[j+1]:,[0,1,2,3]].reshape((-1,4)),plot_datapoints=False,
            color=colors[j+1],levels=[0.68,0.95],fill_contours=True,
            labels=['$H_0$','$\Omega_m$','$w_0$','$w_a$'],
            dpi=300,truths=[70.,0.3,-1.0,0.],truth_color='black',
            fig=figure,label_kwargs={'fontsize':24},smooth=2.)

    custom_lines = [
        Line2D([0], [0], color=colors[0], lw=4),
        Line2D([0], [0], color=colors[j+1], lw=4)]

    custom_labels = [
        'Baseline',
        exp_labels[j]
    ]

    axes = np.array(figure.axes).reshape((4, 4))
    bounds = [[62.,78.],[0.05,0.5],[-2,0.1],[-2,2]]
    for r in range(0,4):
        for c in range(0,r+1):
            if bounds is not None:
                axes[r,c].set_xlim(bounds[c])
                if r != c :
                    axes[r,c].set_ylim(bounds[r])

    #plt.suptitle(exp_names[j],fontsize=16)

    #axes = np.array(figure.axes).reshape((num_params, num_params))
    axes[0,3].legend(custom_lines,custom_labels,frameon=False,fontsize=20)
    plt.savefig('/Users/smericks/Desktop/exp%d.pdf'%(j))

### Time-Delay Exps plot

In [ ]:
test_chain = np.transpose(chains_dict['exp0_2']['w0wa_seed6']['chain'],axes=(1,0,2))
burnin = 20000

chains = [
    np.transpose(chains_dict['exp0_2']['w0wa_seed6_LONG']['chain'][:70000],axes=(1,0,2)),
    np.transpose(chains_dict['exp5_1']['w0wa_seed6_LONG']['chain'][:70000],axes=(1,0,2)),
    np.transpose(chains_dict['exp5_2']['w0wa_seed6_LONG']['chain'][:70000],axes=(1,0,2)),
    np.transpose(chains_dict['exp5_3']['w0wa_seed6_LONG']['chain'][:70000],axes=(1,0,2)),
    np.transpose(chains_dict['exp5_4']['w0wa_seed6_LONG']['chain'][:70000],axes=(1,0,2))
]

labels = ['Baseline',
          '+70 $\sigma(\Delta t)$= 2 days',
          '$\sigma(\Delta t)_{LSST}$= 4 days',
           '$\sigma(\Delta t)_{LSST}$= 3 days',
            '$\sigma(\Delta t)_{LSST}$= 2 days']


colors = ['darkgrey','cornflowerblue',"#BFA6D7","#A06AD2",'#8A2BE2']
plt.figure(dpi=200)
for j,curr_chain in enumerate(chains):
    print(labels[j])
    test_chain = curr_chain[:,burnin:,:].reshape((-1,curr_chain.shape[2]))
    #sigma_h0 = np.std(test_chain[:,0],ddof=1)
    arviz_hdi = az.hdi(test_chain[:,0], hdi_prob=.68)
    sigma_h0 = (arviz_hdi[1] - arviz_hdi[0])/2
    zp,de_fom = DE_fom(curr_chain,burnin)

    plt.scatter(1 / sigma_h0 ,de_fom,label=labels[j],s=200,c=colors[j])
    #plt.scatter(100 * sigma_h0 / np.median(test_chain[:,0]),de_fom,label=labels[j],s=120,c=colors[j])

plt.legend(loc='lower right',fontsize=12)
plt.xlabel(r'1 / $\sigma(H_0)$',fontsize=20)
plt.ylabel('DE FOM [$\sigma(w_0) \sigma(w_p)$]$^{-1}$',fontsize=17)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.savefig('/Users/smericks/Desktop/time_delay_discussion.pdf',bbox_inches='tight')

## Figure out why adding in FM doesn't change anything?? ##

In [ ]:
exp0_2_datavector_path = 'InferenceRuns/exp0_2/static_datavectors_seed6.json'
exp4_2_datavector_path = 'InferenceRuns/exp4_2/static_datavectors_seed6.json'

def retrieve_dv(static_dv_filepath):

    # load in static data vectors
    with open(static_dv_filepath, 'r') as file:
        data_vector_dict_list = json.load(file)

    # return it to np.array
    for dv_dict in data_vector_dict_list:
        for key in dv_dict.keys():
            if isinstance(dv_dict[key], list):
                dv_dict[key] = np.asarray(dv_dict[key])

    return data_vector_dict_list

exp0_2_datavectors = retrieve_dv(exp0_2_datavector_path)
exp4_2_datavectors = retrieve_dv(exp4_2_datavector_path)

In [ ]:
# 4th object in the dv_list should have diff precision

old_fpd_samps = exp0_2_datavectors[3]['fpd_samples'] 
old_lensparam_samps = exp0_2_datavectors[3]['lens_param_samples'] 
old_kin_samps = exp0_2_datavectors[3]['kin_pred_samples'] 

new_fpd_samps = exp4_2_datavectors[3]['fpd_samples']
new_lensparam_samps = exp4_2_datavectors[3]['lens_param_samples'] 
new_kin_samps = exp4_2_datavectors[3]['kin_pred_samples'] 

plt.figure(dpi=150)
plt.hist(old_fpd_samps[10,:,0],density=True,histtype='step',label='exp0_2')
plt.hist(new_fpd_samps[10,:,0],density=True,histtype='step',label='exp4_2')
plt.legend()
plt.title('Aperture-Kin Lens 10, fpd01')

plt.figure(dpi=150)
plt.hist(old_kin_samps[10,:,0],density=True,histtype='step',label='exp0_2')
plt.hist(new_kin_samps[10,:,0],density=True,histtype='step',label='exp4_2')
plt.legend()
plt.title('Aperture-Kin Lens 10, csqrtJ')

print(new_kin_samps[10,:,0].shape)

## Time-Delay Exp ##

In [ ]:
import matplotlib

exp_chains = [
    np.transpose(chains_dict['exp0_2']['w0wa_seed4_TdcosmoPrior']['chain'][:70000],axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed4_10k_fpd']['chain'][:70000],axes=(1,0,2))]
exp_names = [
             'Exp 0.2: Seed 3, 5k fpd samps',
             'Exp 0.2: Seed 3, 10k fpd samps']

num_chains = len(exp_chains)
burnin = [20000]*num_chains
colors = ['C3','darkred']

truth_colors = ["#000000"] * num_chains

custom_lines = []
custom_labels = []

for i,exp_chain in enumerate(exp_chains):

    num_params = exp_chain.shape[2]

    my_color = colors[i]
    
    print(exp_names[i])
    #median_and_uncertainty(exp_chain,burnin[i])
    print(exp_names[i])
    zp,fom = DE_fom(exp_chain,burnin[i])
     
    if i ==0:

        figure = corner.corner(exp_chain[:,burnin[i]:,:-2].reshape((-1,exp_chain.shape[2]-2)),plot_datapoints=False,
            color=my_color,levels=[0.68,0.95],fill_contours=True,
            labels= ['$H_0$','$\Omega_M$','$w_0$','$w_a$',
                r'$\mu(\lambda_{int})$',r'$\sigma(\lambda_{int})$',
                r'$\mu(\beta_{ani})$',r'$\sigma(\beta_{ani})$'],
            dpi=300,truths=[70.,0.3,-1.0,0.,1.,0.1,0.,0.1],truth_color=truth_colors[i],
            fig=None,label_kwargs={'fontsize':24},
            smooth=3)

    else:

        corner.corner(exp_chain[:,burnin[i]:,:-2].reshape((-1,exp_chain.shape[2]-2)),plot_datapoints=False,
            color=my_color,levels=[0.68,0.95],fill_contours=True,
            labels=['$H_0$','$\Omega_M$','$w_0$','$w_a$',
                r'$\mu(\lambda_{int})$',r'$\sigma(\lambda_{int})$',
                r'$\mu(\beta_{ani})$',r'$\sigma(\beta_{ani})$'],
            dpi=300,truths=[70.,0.3,-1.0,0.,1.,0.1,0.,0.1],truth_color=truth_colors[i],
            fig=figure,label_kwargs={'fontsize':24},smooth=3)
        
    custom_lines.append(Line2D([0], [0], color=my_color, lw=4))

    # calculate h0 constraint
    h0, h0_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,0].reshape((-1,1)),weights=None)
    OmegaM, OmegaM_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,1].reshape((-1,1)),weights=None)
    w0, w0_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,2].reshape((-1,1)),weights=None)
    wa, wa_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,3].reshape((-1,1)),weights=None)
    # construct label
    custom_labels.append(exp_names[i]+
        ':\n $H_0$=%.2f$\pm$%.2f \n $\Omega_M$=%.2f$\pm$%.2f \n DE FOM = %.2f'%(
        h0, h0_sigma, OmegaM, OmegaM_sigma, fom))

"""
axes = np.array(figure.axes).reshape((3, 3))
bounds = [[63,77],[1.91,2.095],[0.0,0.2]]
for r in range(0,3):
        for c in range(0,r+1):
            if bounds is not None:
                axes[r,c].set_xlim(bounds[c])
                if r != c :
                    axes[r,c].set_ylim(bounds[r])

axes = np.array(figure.axes).reshape((3, 3))
"""

axes = np.array(figure.axes).reshape((8, 8))
axes[0,7].legend(custom_lines,custom_labels,frameon=False,fontsize=30)
plt.savefig('/Users/smericks/Desktop/lens_selection_comp.pdf')

In [ ]:
import matplotlib

exp_chains = [
    np.transpose(chains_dict['exp0_2']['w0wa_seed6_LONG']['chain'][:70000],axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed6_GAMMA']['chain'][:70000],axes=(1,0,2))]
exp_names = [
             'Exp 0.2: Baseline',
             'Exp 0.2: Baseline + $\mu(\gamma_{lens})$']

num_chains = len(exp_chains)
burnin = [20000]*num_chains
cmap = plt.get_cmap('ocean')
colors = ['darkgrey','orangered','deepskyblue']
truth_colors = ["#000000"] * num_chains

custom_lines = []
custom_labels = []

for i,exp_chain in enumerate(exp_chains):

    num_params = exp_chain.shape[2]

    my_color = colors[i]
    
    print(exp_names[i])
    #median_and_uncertainty(exp_chain,burnin[i])
    print(exp_names[i])
    zp,fom = DE_fom(exp_chain,burnin[i])
     
    if i ==0:

        np.save('lensed_AGN_samps.npy',
            exp_chain[:,burnin[i]:].reshape((-1,exp_chain.shape[2])) )

        figure = corner.corner(exp_chain[:,burnin[i]:].reshape((-1,exp_chain.shape[2])),plot_datapoints=False,
            color=my_color,levels=[0.68,0.95],fill_contours=True,
            labels= ['$H_0$','$\Omega_M$','$w_0$','$w_a$',
                r'$\mu(\lambda_{int})$',r'$\sigma(\lambda_{int})$',
                r'$\mu(\beta_{ani})$',r'$\sigma(\beta_{ani})$',
                r'$\mu(\gamma_{lens})$',r'$\sigma(\gamma_{lens})$'],
            dpi=300,truths=[70.,0.3,-1.0,0.,1.,0.1,0.,0.1,2.05,0.13],
            #0.,0.1,2.05,0.13],truth_color=truth_colors[i],
            fig=None,label_kwargs={'fontsize':24},smooth=3)

    else:

        corner.corner(exp_chain[:,burnin[i]:].reshape((-1,exp_chain.shape[2])),plot_datapoints=False,
            color=my_color,levels=[0.68,0.95],fill_contours=True,
            labels=['$H_0$','$\Omega_M$','$w_0$','$w_a$',
                r'$\mu(\lambda_{int})$',r'$\sigma(\lambda_{int})$',
                r'$\mu(\beta_{ani})$',r'$\sigma(\beta_{ani})$',
                r'$\mu(\gamma_{lens})$',r'$\sigma(\gamma_{lens})$'],
            dpi=300,truths=[70.,0.3,-1.0,0.,1.,0.1,0.,0.1,2.05,0.13],truth_color=truth_colors[i],
            fig=figure,label_kwargs={'fontsize':24},smooth=3)
        
    custom_lines.append(Line2D([0], [0], color=my_color, lw=4))

    # calculate h0 constraint
    h0, h0_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,0].reshape((-1,1)),weights=None)
    OmegaM, OmegaM_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,1].reshape((-1,1)),weights=None)
    w0, w0_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,2].reshape((-1,1)),weights=None)
    wa, wa_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,3].reshape((-1,1)),weights=None)
    # construct label
    custom_labels.append(exp_names[i]+
        ':\n $H_0$=%.2f$\pm$%.2f \n $\Omega_M$=%.2f$\pm$%.2f \n DE FOM = %.2f'%(
        h0, h0_sigma, OmegaM, OmegaM_sigma, fom))

"""
axes = np.array(figure.axes).reshape((3, 3))
bounds = [[63,77],[1.91,2.095],[0.0,0.2]]
for r in range(0,3):
        for c in range(0,r+1):
            if bounds is not None:
                axes[r,c].set_xlim(bounds[c])
                if r != c :
                    axes[r,c].set_ylim(bounds[r])

axes = np.array(figure.axes).reshape((3, 3))
"""

axes = np.array(figure.axes).reshape((10, 10))
axes[0,8].legend(custom_lines,custom_labels,frameon=False,fontsize=30)
plt.savefig('/Users/smericks/Desktop/lens_selection_comp.pdf')

In [ ]:
import matplotlib

exp_chains = [
    np.transpose(chains_dict['exp1_3']['w0wa_seed1_widerw0wa']['chain'][:50000],axes=(1,0,2)),
    np.transpose(chains_dict['exp1_3']['w0wa_seed2_widerw0wa']['chain'][:50000],axes=(1,0,2)),
    np.transpose(chains_dict['exp1_3']['w0wa_seed3_widerw0wa']['chain'][:50000],axes=(1,0,2)),
    np.transpose(chains_dict['exp1_3']['w0wa_seed4_widerw0wa']['chain'][:50000],axes=(1,0,2)),
    #np.transpose(chains_dict['exp1_3']['w0wa_seed6_widerw0wa']['chain'][:50000],axes=(1,0,2)),
    np.transpose(chains_dict['exp1_3']['w0wa_seed7_widerw0wa']['chain'][:50000],axes=(1,0,2))
    ]

exp_names = ['Exp 1.3: Seed 0',
             'Exp 1.3: Seed 1',
             'Exp 1.3: Seed 2',
             'Exp 1.3: Seed 3',
             #'Exp 1.3: Seed 5',
             'Exp 1.3: Seed 6',
             ]

num_chains = len(exp_chains)
burnin = [20000]*num_chains
cmap = plt.get_cmap('ocean')
colors = ['C0','C1','C2','C3','C6','C7']
truth_colors = ["#000000"] * num_chains

custom_lines = []
custom_labels = []

for i,exp_chain in enumerate(exp_chains):

    num_params = exp_chain.shape[2]

    my_color = colors[i]
    
    print(exp_names[i])
    #median_and_uncertainty(exp_chain,burnin[i])
    print(exp_names[i])
    zp,fom = DE_fom(exp_chain,burnin[i])
     
    if i ==0:

        figure = corner.corner(exp_chain[:,burnin[i]:,:-2].reshape((-1,exp_chain.shape[2]-2)),plot_datapoints=False,
            color=my_color,levels=[0.68,0.95],fill_contours=True,
            labels= ['$H_0$','$\Omega_M$','$w_0$','$w_a$',
                r'$\mu(\lambda_{int})$',r'$\sigma(\lambda_{int})$',
                r'$\mu(\beta_{ani})$',r'$\sigma(\beta_{ani})$'],
            dpi=300,truths=[70.,0.3,-1.0,0.,1.,0.1,0.,0.1],truth_color=truth_colors[i],
            fig=None,label_kwargs={'fontsize':24},
            smooth=3)

    else:

        corner.corner(exp_chain[:,burnin[i]:,:-2].reshape((-1,exp_chain.shape[2]-2)),plot_datapoints=False,
            color=my_color,levels=[0.68,0.95],fill_contours=True,
            labels=['$H_0$','$\Omega_M$','$w_0$','$w_a$',
                r'$\mu(\lambda_{int})$',r'$\sigma(\lambda_{int})$',
                r'$\mu(\beta_{ani})$',r'$\sigma(\beta_{ani})$'],
            dpi=300,truths=[70.,0.3,-1.0,0.,1.,0.1,0.,0.1],truth_color=truth_colors[i],
            fig=figure,label_kwargs={'fontsize':24},smooth=3)
        
    custom_lines.append(Line2D([0], [0], color=my_color, lw=4))

    # calculate h0 constraint
    h0, h0_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,0].reshape((-1,1)),weights=None)
    OmegaM, OmegaM_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,1].reshape((-1,1)),weights=None)
    w0, w0_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,2].reshape((-1,1)),weights=None)
    print('w0 median: ', w0)
    wa, wa_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,3].reshape((-1,1)),weights=None)
    print('wa median: ', wa)

    # construct label
    custom_labels.append(exp_names[i]+
        ':\n $H_0$=%.2f$\pm$%.2f \n $\Omega_M$=%.2f$\pm$%.2f \n DE FOM = %.2f'%(
        h0, h0_sigma, OmegaM, OmegaM_sigma, fom))

"""
axes = np.array(figure.axes).reshape((3, 3))
bounds = [[63,77],[1.91,2.095],[0.0,0.2]]
for r in range(0,3):
        for c in range(0,r+1):
            if bounds is not None:
                axes[r,c].set_xlim(bounds[c])
                if r != c :
                    axes[r,c].set_ylim(bounds[r])

axes = np.array(figure.axes).reshape((3, 3))
"""

axes = np.array(figure.axes).reshape((8, 8))
axes[0,7].legend(custom_lines,custom_labels,frameon=False,fontsize=30)
plt.savefig('/Users/smericks/Desktop/lens_selection_comp.pdf')

Need a new way to check chain convergence...

In [ ]:
# let's do chunks of 500 after burnin=2000
def plot_posterior_median(emcee_chain,burnin,param_idx,color='black',figure=None,label=''):
    """
    Args:
        emcee_chain (shape (n_steps,n_walkers,n_params))
    """
    chunk_size = 1000
    min_idx = burnin
    max_idx = burnin+chunk_size

    if figure is None:
        figure = plt.figure(dpi=200)
        gs = figure.add_gridspec(3, 1, height_ratios=[3, 1, 1])
        ax_main = figure.add_subplot(gs[0])
        figure.add_subplot(gs[1], sharex=ax_main)
        figure.add_subplot(gs[2], sharex=ax_main)

    mean_vals = []
    std_dev_vals = []
    x_vals = []

    while max_idx < np.shape(emcee_chain)[0]:
        chunk_of_samps = emcee_chain[min_idx:max_idx,:,param_idx]
        mean = np.mean(chunk_of_samps)
        std_dev = np.std(chunk_of_samps,ddof=1)
        
        # Track and plot mean and std_dev in bottom panels
        mean_vals.append(mean)
        std_dev_vals.append(std_dev)
        x_vals.append(max_idx)

        min_idx += chunk_size
        max_idx += chunk_size

    # Add bottom panels for mean and std_dev
    axs = [figure.axes[0], figure.axes[1], figure.axes[2]]
    mean_vals = np.array(mean_vals)
    std_dev_vals = np.array(std_dev_vals)
    x_vals_arr = np.array(x_vals)
    axs[0].plot(x_vals_arr, mean_vals, color=color,label=label)
    axs[0].fill_between(x_vals_arr, mean_vals - std_dev_vals, mean_vals + std_dev_vals, color=color, alpha=0.3)
    last_5 = np.mean(mean_vals)
    axs[1].plot(x_vals,(mean_vals - last_5),color=color)
    axs[1].hlines(y=0., xmin=x_vals_arr[0], xmax=x_vals_arr[-1], color="#FF5050", linestyles='--', linewidth=1)
    axs[1].set_ylabel('$\Delta \mu$')
    last_5 = np.mean(std_dev_vals)
    axs[2].plot(x_vals,std_dev_vals - last_5,color=color)
    axs[2].hlines(y=0., xmin=x_vals_arr[0], xmax=x_vals_arr[-1], color="#FF5050", linestyles='--', linewidth=1)
    axs[2].set_ylabel('$\Delta \sigma$')
    plt.tight_layout()

    return figure

#colors = ['darkgrey','seagreen','yellowgreen', 'cornflowerblue','blueviolet']
figure2 = plot_posterior_median(chains_dict['exp0_2']['w0wa_seed3_TdcosmoPrior']['chain'],500,param_idx=1,
    color='C2',label=r'Exp 0.2, Seed 2, 5k fpd steps')
figure2 = plot_posterior_median(chains_dict['exp0_2']['w0wa_seed3_10k_fpd']['chain'],500,param_idx=1,
    color='darkgreen',label=r'Exp 0.2, Seed 2, 10k fpd steps',figure=figure2)
#figure2.axes[0].hlines(y=70.,xmin=1500,xmax=50000,color='black',label='Truth')
figure2.axes[0].legend(fontsize=8,loc='lower left')
figure2.axes[0].set_title(r'$\Omega_M$')

# 'seagreen','yellowgreen'

### Let's evaluate stochasticity across seeds ###

In [ ]:
ten_seeds = [
    np.transpose(chains_dict['exp0_2']['w0wa_seed1']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed2']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed3']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed4']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed5']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed6']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed7']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed8']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed9']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed10']['chain'], axes=(1,0,2))
]
plt.figure(dpi=200)
burnin = 30000
truth_vals = [70.,0.3,-1.,0.,1.,0.1,0.,0.1]
for i,chain in enumerate(ten_seeds):

    samps = chain[:,burnin:,:].reshape((-1,exp_chain.shape[2]))
    samps = samps[:,:8]
    med = np.median(samps,axis=0)
    low = np.quantile(samps,q=0.1586,axis=0)
    high = np.quantile(samps,q=0.8413,axis=0)

    error = med - truth_vals
    sigma = ((high-med)+(med-low))/2
    bias = error/sigma

    plt.plot([0,1,2,3,4,5,6,7], bias,label='Seed %d'%(i))
    if i == 5:
        plt.scatter([0,1,2,3,4,5,6,7], bias, s=100, marker='*',edgecolors='peru',zorder=200)
    else:
        plt.scatter([0,1,2,3,4,5,6,7], bias, s=50)

plt.hlines(y=0.,xmin=0.,xmax=7.,color='black')
plt.ylabel('Bias (in $\sigma$)',fontsize=15)
plt.xticks(ticks=[0,1,2,3,4,5,6,7],labels=['$H_0$','$\Omega_M$','$w_0$','$w_a$',
    r'$\mu(\lambda_{int})$',r'$\sigma(\lambda_{int})$',
    r'$\mu(\beta_{ani})$',r'$\sigma(\beta_{ani})$'],fontsize=13)

plt.axhspan(-1, 1, color='grey', alpha=0.2)
#plt.legend(loc='lower left',fontsize=6)
#plt.title('Experiment 0.2, Ten Random Seeds')
plt.savefig('/Users/smericks/Desktop/baseline_fluctuation_bias.pdf')

In [ ]:
burnin=20000
chains = [
    np.transpose(chains_dict['exp0_2']['w0wa_seed1_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed2_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed3_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed4_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed5_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed6_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed7_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed8_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed9_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed10_pantheonOM']['chain'], axes=(1,0,2))
]
"""
chains = [
    np.transpose(chains_dict['exp1_3']['w0wa_seed1_OmegaM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp1_3']['w0wa_seed2_OmegaM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp1_3']['w0wa_seed3_OmegaM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp1_3']['w0wa_seed4_OmegaM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp1_3']['w0wa_seed5_OmegaM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp1_3']['w0wa_seed6_OmegaM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp1_3']['w0wa_seed7_OmegaM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp1_3']['w0wa_seed8_OmegaM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp1_3']['w0wa_seed9_OmegaM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp1_3']['w0wa_seed10_OmegaM']['chain'], axes=(1,0,2))
]
"""
de_fom_list = []
plt.figure(dpi=200)
for j,curr_chain in enumerate(chains):
    print(curr_chain.shape)
    test_chain = curr_chain[:,burnin:,:].reshape((-1,curr_chain.shape[2]))
    #sigma_h0 = np.std(test_chain[:,0],ddof=1)
    arviz_hdi = az.hdi(test_chain[:,0], hdi_prob=.68)
    sigma_h0 = (arviz_hdi[1] - arviz_hdi[0])/2
    arviz_hdi = az.hdi(test_chain[:,1], hdi_prob=.68)
    sigma_omegaM = (arviz_hdi[1] - arviz_hdi[0])/2
    # check median w0
    median_w0 = np.median(test_chain[:,2])
    print('seed %d'%(j))
    print('sigma h0: ', sigma_h0)
    print('percent h0: ', 100*sigma_h0/np.median(test_chain[:,0]))
    print('sigma omegaM: ', sigma_omegaM)
    zp,de_fom = DE_fom(curr_chain,burnin)
    de_fom_list.append(de_fom)
    #if j == 5:
    #    plt.scatter(sigma_h0,de_fom,label='Init seed %d'%(j),s=180,marker='*',edgecolors='peru')
    #else:
    plt.scatter(median_w0,de_fom,label='Init seed %d'%(j),s=90)

plt.ylabel('DE FOM [$\sigma(w_0) \sigma(w_p)$]$^{-1}$',fontsize=15)
plt.xlabel('$w_0$',fontsize=17)
#plt.legend(fontsize=12)
#plt.xlim([1.4,2.1])
plt.ylim([2.,22.])
plt.title('Pantheon+ $\Omega_m$ prior')
plt.savefig('/Users/smericks/Desktop/baseline_fluctuation_precision.pdf')

In [ ]:
from astropy.cosmology import w0waCDM

burnin=20000
chains = [
    np.transpose(chains_dict['exp0_2']['w0wa_seed1_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed2_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed3_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed4_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed5_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed6_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed7_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed8_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed9_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed10_pantheonOM']['chain'], axes=(1,0,2))
]

h0_input = 70.
omega_m_input = 0.3
omega_de_input = 1. - omega_m_input

w0_input = -1.
wa_input = 0.

w0_vals = [-1,-1,-1.,-1.,-1.,-1,-1]
wa_vals = [-10,-4.,-2,0.,2,4.,10]

z_range = np.arange(1100,-1.,-0.1)

scale_factors = []
ages = []
present_ages = []
comoving_distances = []

for curr_chain in chains:
    test_chain = curr_chain[:,burnin:,:].reshape((-1,curr_chain.shape[2]))
    
    h0_input = np.median(test_chain[:,0])
    omega_m_input = np.median(test_chain[:,1])
    omega_de_input = 1. - omega_m_input
    w0_input = np.median(test_chain[:,2])
    wa_input = np.median(test_chain[:,3])

    w0wa_cosmo = w0waCDM(H0=h0_input,
                    Om0=omega_m_input,Ode0=omega_de_input,
                    w0=w0_input,wa=wa_input)

    scale_factors.append(w0wa_cosmo.scale_factor(z_range))
    ages.append(w0wa_cosmo.age(z_range))
    present_ages.append(w0wa_cosmo.age(0.))
    comoving_distances.append(w0wa_cosmo.comoving_distance(z_range))

# append the ground truth!
w0wa_cosmo = w0waCDM(H0=70.,
                Om0=0.3,Ode0=(1-0.3),
                w0=-1.,wa=0.)

scale_factors.append(w0wa_cosmo.scale_factor(z_range))
ages.append(w0wa_cosmo.age(z_range))
present_ages.append(w0wa_cosmo.age(0.))
comoving_distances.append(w0wa_cosmo.comoving_distance(z_range))


#labels = ['']
colors = ['C0','C1','C2','C3','C4','C5','C6','C7','C8','C9','black']

fig,axs = plt.subplots(1,2,dpi=200,figsize=(12,4))
for i in range(0,len(ages)):
    axs[0].plot(z_range,comoving_distances[i],color=colors[i])
    #axs[0].vlines(x=present_ages[i].value, ymin=0, ymax=1.25, color=colors[i], linestyle='--',alpha=0.5)
    axs[1].plot(z_range,comoving_distances[i],color=colors[i])
    
axs[0].set_xlabel('Redshift')
axs[0].set_ylabel('Comoving Distance (Mpc)')
axs[0].set_xlim([0,3.])
axs[0].set_ylim([0,5400])
axs[1].set_xlabel('Redshift')
axs[1].set_ylabel('Comoving Distance (Mpc)')
axs[1].set_xlim([0.5,1.])
axs[1].set_ylim([0,5400])

In [ ]:
#labels = ['']
colors = ['C0','C1','C2','C3','C4','C5','C6','C7','C8','C9','black']

fig,axs = plt.subplots(1,2,dpi=200,figsize=(12,4))
for i in range(0,len(ages)):
    axs[0].plot(z_range,comoving_distances[i]-comoving_distances[-1],color=colors[i])
    #axs[0].vlines(x=present_ages[i].value, ymin=0, ymax=1.25, color=colors[i], linestyle='--',alpha=0.5)
    axs[1].plot(z_range,comoving_distances[i]-comoving_distances[-1],color=colors[i])
    
axs[0].set_xlabel('Redshift')
axs[0].set_ylabel('$\Delta$ Comoving Distance (Mpc)')
axs[0].set_xlim([0.,1.5])
axs[0].set_ylim([-60,60])
axs[1].set_xlabel('Redshift')
axs[1].set_ylabel('$\Delta$ Comoving Distance (Mpc)')
axs[1].set_xlim([0,3.])
axs[1].set_ylim([-180,180])

In [ ]:
test_chain = np.asarray([tdcosmo2025_cosmo_only_chain])
print(test_chain.shape)
plt.scatter(np.arange(0,500000),test_chain[0,:500000,0])
plt.title('TDCOSMO $H_0$ Chain')
plt.xlim([200000,500000])
plt.ylim([60,70])

In [ ]:
test_chain = np.transpose(chains_dict['exp0_2']['w0wa_shajib_prior_seed6']['chain'], axes=(1,0,2))
print(test_chain.shape)
for j in [0,10,20,30,40]:
    plt.figure()
    for i in range(0,1):
        plt.scatter(np.arange(0,10),test_chain[i+j,:10,3])
        #plt.scatter(np.arange(0,10000)+60000,test_chain[i+j,60000:70000,1])
    plt.title('$w_a$ Chains %d - %d'%(j,j+9))


## TODO: calculate fpd precision for every lens in HST-FM, HST-NPE ## 

In [ ]:
truth_df = pd.read_csv('DataVectors/gold/truth_metadata.csv')

# JWST-FM, HST-FM, HST-NPE, LSST-NPE
dbl_dv_files = ['DataVectors/gold/dbl_posteriors_JWST_DEBIASED.h5',
                'DataVectors/gold/dbl_posteriors_TDCOSMO25_DEBIASED.h5',
                'DataVectors/gold/dbl_posteriors_DEBIASED.h5',
                'DataVectors/silver/dbl_posteriors_DEBIASED.h5']
quad_dv_files = ['DataVectors/gold/quad_posteriors_JWST_DEBIASED.h5',
                'DataVectors/gold/quad_posteriors_TDCOSMO25_DEBIASED.h5',
                'DataVectors/gold/quad_posteriors_DEBIASED.h5',
                'DataVectors/silver/quad_posteriors_DEBIASED.h5']

model_type = ['JWST-FM','HST-FM','HST-NPE','LSST-NPE']

plt.figure()

for j,mt in enumerate(model_type):

    fpd_perc_prec = []

    # dbls
    with h5py.File(dbl_dv_files[j],'r') as h5:
        fpd_samps = h5['fpd_samps'][:]
        lens_param_samps = h5['lens_param_samps'][:]
        catalog_idxs = h5['catalog_idxs'][:]
        for i,cidx in enumerate(catalog_idxs):
            truth_row = truth_df[np.isin(truth_df['catalog_idx'],cidx)]
            fpd_perc_prec.append( np.std(fpd_samps[i,:,0]) / np.abs(truth_row['fpd01'].item()) )

    # quads
    with h5py.File(quad_dv_files[j],'r') as h5:
        fpd_samps = h5['fpd_samps'][:]
        lens_param_samps = h5['lens_param_samps'][:]
        catalog_idxs = h5['catalog_idxs'][:]
        for i,cidx in enumerate(catalog_idxs):
            truth_row = truth_df[np.isin(truth_df['catalog_idx'],cidx)]
            fpd_perc_prec.append( np.std(fpd_samps[i,:,0]) / np.abs(truth_row['fpd01'].item()) )

    fpd_perc_prec = np.asarray(fpd_perc_prec)
    print(mt,' ',np.median(fpd_perc_prec))
    plt.hist(fpd_perc_prec[fpd_perc_prec < 1.],histtype='step',label=mt)

plt.legend()

In [ ]:
import arviz as az
burnin=20000
chains = [
    #np.transpose(chains_dict['exp1_3']['w0wa_seed1_OmegaM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp1_3']['w0wa_seed2_OmegaM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp1_3']['w0wa_seed3_OmegaM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp1_3']['w0wa_seed4_OmegaM']['chain'], axes=(1,0,2)),
    #np.transpose(chains_dict['exp1_3']['w0wa_seed5_OmegaM']['chain'], axes=(1,0,2)),
    #np.transpose(chains_dict['exp1_3']['w0wa_seed6_OmegaM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp1_3']['w0wa_seed7_OmegaM']['chain'], axes=(1,0,2)),
    #np.transpose(chains_dict['exp1_3']['w0wa_seed8_OmegaM']['chain'], axes=(1,0,2)),
    #np.transpose(chains_dict['exp1_3']['w0wa_seed9_OmegaM']['chain'], axes=(1,0,2)),
    #np.transpose(chains_dict['exp1_3']['w0wa_seed10_OmegaM']['chain'], axes=(1,0,2))
]
truth_df = pd.read_csv('DataVectors/gold/truth_metadata.csv')
scatter_colors = ['C1','C2','C3','C6']
scatter_marker = ["1","2","3","4","o"]
seed_nums = [1,2,3,6]

de_fom_list = []
current_param_list = []
plt.figure(dpi=200)
print('len chains',len(chains))
for j,curr_chain in enumerate(chains):

    print('seed %d'%(seed_nums[j]))
    all_cidx = np.load('InferenceRuns/exp0_2/catalog_idxs_seed%d.npy'%(seed_nums[j]))
    print('jwst cidx: ', all_cidx[:10])

    cidx_gold = all_cidx[30:50]

    # INDEXING KEEPING ORDERING THE EXACT SAME (for things that aren't means!)
    fpd_prec_in_order = []
    longest_td_in_order = []
    z_lens_in_order = []
    z_src_in_order = []
    gamma_in_order = []
    # MUSE QUADS
    with h5py.File('DataVectors/gold/quad_posteriors_TDCOSMO25_DEBIASED.h5','r') as h5:
        fpd_samps = h5['fpd_samps'][:]
        lens_param_samps = h5['lens_param_samps'][:]
        catalog_idxs = h5['catalog_idxs'][:]
        for cidx in all_cidx[10:30]:
            # find index of this cidx
            h5_idx = np.where(catalog_idxs==cidx)[0]
            # pull the fpd precision
            fpd_sigma = np.std(fpd_samps[h5_idx,:,2])
            # pull the truth fpd
            truth_row = truth_df[np.isin(truth_df['catalog_idx'],cidx)]
            # tabulate
            fpd_prec_in_order.append(fpd_sigma / np.abs(truth_row['fpd03'].item()))
            longest_td_in_order.append(np.abs(truth_row['td03'].item()))
            gamma_in_order.append(truth_row['main_deflector_parameters_gamma'].item())
            z_lens_in_order.append(truth_row['lens_light_parameters_z_source'].item())
            z_src_in_order.append(truth_row['source_parameters_z_source'].item())

    # MUSE DOUBLES
    with h5py.File('DataVectors/gold/dbl_posteriors_TDCOSMO25_DEBIASED.h5','r') as h5:
        fpd_samps = h5['fpd_samps'][:]
        lens_param_samps = h5['lens_param_samps'][:]
        catalog_idxs = h5['catalog_idxs'][:]
        for cidx in all_cidx[30:50]:
            # find index of this cidx
            h5_idx = np.where(catalog_idxs==cidx)[0]
            # pull the fpd precision
            fpd_sigma = np.std(fpd_samps[h5_idx,:,0])
            # pull the truth fpd
            truth_row = truth_df[np.isin(truth_df['catalog_idx'],cidx)]
            # tabulate
            fpd_prec_in_order.append(fpd_sigma / np.abs(truth_row['fpd01'].item()))
            longest_td_in_order.append(np.abs(truth_row['td01'].item()))
            gamma_in_order.append(truth_row['main_deflector_parameters_gamma'].item())
            z_lens_in_order.append(truth_row['lens_light_parameters_z_source'].item())
            z_src_in_order.append(truth_row['source_parameters_z_source'].item())

    # 4MOST QUADS
    with h5py.File('DataVectors/gold/quad_posteriors_DEBIASED.h5','r') as h5:
        fpd_samps = h5['fpd_samps'][:]
        lens_param_samps = h5['lens_param_samps'][:]
        catalog_idxs = h5['catalog_idxs'][:]
        for cidx in all_cidx[50:125]:
            # find index of this cidx
            h5_idx = np.where(catalog_idxs==cidx)[0]
            # pull the fpd precision
            fpd_sigma = np.std(fpd_samps[h5_idx,:,2])
            # pull the truth fpd
            truth_row = truth_df[np.isin(truth_df['catalog_idx'],cidx)]
            # tabulate
            fpd_prec_in_order.append(fpd_sigma / np.abs(truth_row['fpd03'].item()))
            longest_td_in_order.append(np.abs(truth_row['td03'].item()))
            gamma_in_order.append(truth_row['main_deflector_parameters_gamma'].item())
            z_lens_in_order.append(truth_row['lens_light_parameters_z_source'].item())
            z_src_in_order.append(truth_row['source_parameters_z_source'].item())

    # 4MOST DOUBLES
    with h5py.File('DataVectors/gold/dbl_posteriors_DEBIASED.h5','r') as h5:
        fpd_samps = h5['fpd_samps'][:]
        lens_param_samps = h5['lens_param_samps'][:]
        catalog_idxs = h5['catalog_idxs'][:]
        for cidx in all_cidx[125:200]:
            # find index of this cidx
            h5_idx = np.where(catalog_idxs==cidx)[0]
            # pull the fpd precision
            fpd_sigma = np.std(fpd_samps[h5_idx,:,0])
            # pull the truth fpd
            truth_row = truth_df[np.isin(truth_df['catalog_idx'],cidx)]
            # tabulate
            fpd_prec_in_order.append(fpd_sigma / np.abs(truth_row['fpd01'].item()))
            longest_td_in_order.append(np.abs(truth_row['td01'].item()))
            gamma_in_order.append(truth_row['main_deflector_parameters_gamma'].item())
            z_lens_in_order.append(truth_row['lens_light_parameters_z_source'].item())
            z_src_in_order.append(truth_row['source_parameters_z_source'].item())

    # LSST Portion of Sample
    with h5py.File('DataVectors/silver/dbl_posteriors_DEBIASED.h5','r') as h5:
        dbl_fpd_samps = h5['fpd_samps'][:]
        dbl_lens_param_samps = h5['lens_param_samps'][:]
        dbl_catalog_idxs = h5['catalog_idxs'][:]
    with h5py.File('DataVectors/silver/quad_posteriors_DEBIASED.h5','r') as h5:
        quad_fpd_samps = h5['fpd_samps'][:]
        quad_lens_param_samps = h5['lens_param_samps'][:]
        quad_catalog_idxs = h5['catalog_idxs'][:]
    
    for cidx in all_cidx[200:]:
        # pull the truth fpd
        truth_row = truth_df[np.isin(truth_df['catalog_idx'],cidx)]
        # tabulate
        td03 = truth_row['td03'].item()
        # DOUBLE
        if np.isnan(td03):
            # find index of this cidx
            h5_idx = np.where(dbl_catalog_idxs==cidx)[0]
            # pull the fpd precision
            fpd_sigma = np.std(dbl_fpd_samps[h5_idx,:,0])
            fpd_prec_val = fpd_sigma / np.abs(truth_row['fpd01'].item())
            # longest td
            longest_td_val = np.abs(truth_row['td01'].item())

        # QUAD:
        else:
            # find index of this cidx
            h5_idx = np.where(quad_catalog_idxs==cidx)[0]
            # pull the fpd precision
            fpd_sigma = np.std(quad_fpd_samps[h5_idx,:,2])
            fpd_prec_val = fpd_sigma / np.abs(truth_row['fpd03'].item())
            # longest td
            longest_td_val = np.abs(truth_row['td03'].item())

        fpd_prec_in_order.append(fpd_prec_val)
        longest_td_in_order.append(longest_td_val)
        gamma_in_order.append(truth_row['main_deflector_parameters_gamma'].item())
        z_lens_in_order.append(truth_row['lens_light_parameters_z_source'].item())
        z_src_in_order.append(truth_row['source_parameters_z_source'].item())


    # now do an approximate DDT precision
    fpd_prec_in_order = np.asarray(fpd_prec_in_order)
    longest_td_in_order = np.asarray(longest_td_in_order)
    proxy_ddt_prec_in_order = np.empty(len(fpd_prec_in_order))
    proxy_ddt_prec_in_order[:40] = np.sqrt(fpd_prec_in_order[:40]**2 + 0.03**2)        
    proxy_ddt_prec_in_order[40:] = np.sqrt(fpd_prec_in_order[40:]**2 + (5/longest_td_in_order[40:])**2 )     

    # Index into truth_df by catalog_idx
    # gold_df = gold_df[~gold_df['catalog_idx'].isin(nirspec_quads_catalog_idxs)].reset_index(drop=True)
    gold_truth = truth_df[truth_df['catalog_idx'].isin(cidx_gold)].reset_index(drop=True)
    dbls_truth = truth_df[truth_df['catalog_idx'].isin(cidx_ifu_dbls)].reset_index(drop=True)
    z_lens_gold = gold_truth['lens_light_parameters_z_source'].to_numpy()
    mu_z_lens = np.mean(z_lens_gold)
    min_z_lens = np.min(z_lens_gold)
    median_z_lens = np.median(z_lens_gold)

    z_src_gold = gold_truth['source_parameters_z_source'].to_numpy()
    mu_z_src = np.mean(z_src_gold)

    from astropy.cosmology import FlatLambdaCDM
    gt_cosmo = FlatLambdaCDM(H0=70.,Om0=0.3)
    D_ds_gold = np.array(gt_cosmo.angular_diameter_distance_z1z2(z_lens_gold,z_src_gold))
    mean_D_ds = np.mean(D_ds_gold)
    Ddt_Mpc_gold = gold_truth['Ddt_Mpc'].to_numpy()

    theta_E_gold = gold_truth['main_deflector_parameters_theta_E'].to_numpy()
    mu_theta_E = np.mean(theta_E_gold)

    gamma_gold = gold_truth['main_deflector_parameters_gamma'].to_numpy()
    mu_gamma = np.mean(gamma_gold)
    median_gamma = np.median(gamma_gold)

    m_app_ps = gold_truth['point_source_parameters_mag_app'].to_numpy()
    min_pointsource_magapp = np.min(m_app_ps)

    beta_ani_gold = gold_truth['beta_ani'].to_numpy()
    lambda_int_gold = gold_truth['lambda_int'].to_numpy()

    td01 = gold_truth['td01'].to_numpy()
    td02 = gold_truth['td02'].to_numpy()
    td03 = gold_truth['td03'].to_numpy()
    shortest_td_gold = np.abs(td01)
    longest_td = np.abs(td03)
    longest_td[np.isnan(longest_td)] = np.abs(td01[np.isnan(longest_td)])
    mean_longest_td = np.mean(longest_td)
    mean_longest_td_prec = np.mean(100 * 5/longest_td)
    max_longest_td = np.max(longest_td)
    td_longer_than_200 = np.sum(longest_td>200)


    fpd01 = gold_truth['fpd01'].to_numpy()
    fpd02 = gold_truth['fpd02'].to_numpy()
    fpd03 = gold_truth['fpd03'].to_numpy()
    longest_fpd = np.abs(fpd03)
    longest_fpd[np.isnan(longest_fpd)] = np.abs(fpd01[np.isnan(longest_fpd)])
    mean_longest_fpd = np.mean(longest_fpd)

    # lens light properties
    m_app_lens_gold = gold_truth['lens_light_parameters_mag_app'].to_numpy()
    R_sersic_lens_gold = gold_truth['lens_light_parameters_R_sersic'].to_numpy()
    n_sersic_lens_gold = gold_truth['lens_light_parameters_n_sersic'].to_numpy()
    

    test_chain = curr_chain[:,burnin:,:].reshape((-1,curr_chain.shape[2]))
    #sigma_h0 = np.std(test_chain[:,0],ddof=1)
    arviz_hdi = az.hdi(test_chain[:,0], hdi_prob=.68)
    sigma_h0 = (arviz_hdi[1] - arviz_hdi[0])/2
    arviz_hdi = az.hdi(test_chain[:,4], hdi_prob=.68)
    sigma_mu_lint = (arviz_hdi[1] - arviz_hdi[0])/2
    arviz_hdi = az.hdi(test_chain[:,1], hdi_prob=.68)
    sigma_omegaM = (arviz_hdi[1] - arviz_hdi[0])/2
    print('seed %d'%(j))
    print('sigma h0: ', sigma_h0)
    print('sigma omegaM: ', sigma_omegaM)
    zp,de_fom = DE_fom(curr_chain,burnin)
    de_fom_list.append(de_fom)


    lsst_proxy_ddt = proxy_ddt_prec_in_order[200:]
    #num_special = np.sum(lsst_proxy_ddt>0.6)
    num_special = np.sum((z_lens_gold<0.5) & (longest_td > 100.))
    #np.sum((gamma_gold > 2.1) & (longest_td > 50.))
    #num_special = np.sum((gamma_gold > 2.25))

    chosen_param = num_special
    current_param_list.append(chosen_param)
    # TODO: define the bins
    #hist_bins = [0.,0.05,0.1,0.15,0.2,0.25,0.3,0.35,0.4,0.45,0.5,0.55,0.6]
    #hist_bins = [0.02,0.025,0.03,0.035,0.04,0.045,0.05,0.055,0.06,0.065,0.07,0.075,0.08,0.085]#,0.1,0.15,0.2,0.25,0.3,0.35,0.4,0.45,0.5,0.55,0.6]
    hist_bins = np.arange(0.2,2.2,step=0.2)
    #plt.hist(R_sersic_lens_gold,label='seed %d'%(seed_nums[j]),histtype='step',bins=hist_bins,color=scatter_colors[j])
    z_lens_in_order = np.asarray(z_lens_in_order)
    plt.scatter(z_lens_gold,z_src_gold,label='seed %d'%(j),s=80,c=scatter_colors[j],marker=scatter_marker[j],zorder=100*(5-j))
    #if j == 5:
    #    plt.scatter(chosen_param,de_fom,label='seed %d'%(j),s=180,marker='*',edgecolors='peru')
    #else:
    #    plt.scatter(chosen_param,de_fom,label='seed %d'%(j),s=90)

corr_of_param = np.corrcoef(current_param_list,de_fom_list)
#print(corr_of_param)

#plt.ylabel('DE FOM [$\sigma(w_0) \sigma(w_p)$]$^{-1}$',fontsize=15)
plt.xlabel(r'z_lens',fontsize=17)
plt.ylabel(r'z_src',fontsize=17)
#plt.legend(fontsize=12)
plt.title('JWST Quads, Pantheon+ $\Omega_m$ prior')#, corr=%.2f'%(corr_of_param[0,1]))
#plt.ylim([2.,22.])
plt.savefig('/Users/smericks/Desktop/baseline_fluctuation_precision_0Mprior.pdf')

In [ ]:
import arviz as az
burnin=20000
chains = [
    #np.transpose(chains_dict['exp1_3']['w0wa_seed1_OmegaM']['chain'], axes=(1,0,2)),
    #np.transpose(chains_dict['exp1_3']['w0wa_seed2_OmegaM']['chain'], axes=(1,0,2)),
    #np.transpose(chains_dict['exp1_3']['w0wa_seed3_OmegaM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp1_3']['w0wa_seed4_OmegaM']['chain'], axes=(1,0,2)),
    #np.transpose(chains_dict['exp1_3']['w0wa_seed5_OmegaM']['chain'], axes=(1,0,2)),
    #np.transpose(chains_dict['exp1_3']['w0wa_seed6_OmegaM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp1_3']['w0wa_seed7_OmegaM']['chain'], axes=(1,0,2)),
    #np.transpose(chains_dict['exp1_3']['w0wa_seed8_OmegaM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp1_3']['w0wa_seed9_OmegaM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp1_3']['w0wa_seed10_OmegaM']['chain'], axes=(1,0,2))
]
truth_df = pd.read_csv('DataVectors/gold/truth_metadata.csv')
scatter_colors = ['C3','C6','C8','C9']
scatter_marker = ["1","2","3","4","o"]
seed_nums = [3,6,8,9]

de_fom_list = []
current_param_list = []
plt.figure(dpi=200)
print('len chains',len(chains))
for j,curr_chain in enumerate(chains):

    print('seed %d'%(seed_nums[j]))
    all_cidx = np.load('InferenceRuns/exp0_2/catalog_idxs_seed%d.npy'%(seed_nums[j]))
    print('jwst cidx: ', all_cidx[:10])

    cidx_gold = all_cidx[:10]
    cidx_ifu_dbls = all_cidx[30:50]

    # INDEXING KEEPING ORDERING THE EXACT SAME (for things that aren't means!)
    fpd_prec_in_order = []
    longest_td_in_order = []
    z_lens_in_order = []
    z_src_in_order = []
    gamma_in_order = []
    # MUSE QUADS
    with h5py.File('DataVectors/gold/quad_posteriors_TDCOSMO25_DEBIASED.h5','r') as h5:
        fpd_samps = h5['fpd_samps'][:]
        lens_param_samps = h5['lens_param_samps'][:]
        catalog_idxs = h5['catalog_idxs'][:]
        for cidx in all_cidx[10:30]:
            # find index of this cidx
            h5_idx = np.where(catalog_idxs==cidx)[0]
            # pull the fpd precision
            fpd_sigma = np.std(fpd_samps[h5_idx,:,2])
            # pull the truth fpd
            truth_row = truth_df[np.isin(truth_df['catalog_idx'],cidx)]
            # tabulate
            fpd_prec_in_order.append(fpd_sigma / np.abs(truth_row['fpd03'].item()))
            longest_td_in_order.append(np.abs(truth_row['td03'].item()))
            gamma_in_order.append(truth_row['main_deflector_parameters_gamma'].item())
            z_lens_in_order.append(truth_row['lens_light_parameters_z_source'].item())
            z_src_in_order.append(truth_row['source_parameters_z_source'].item())

    # MUSE DOUBLES
    with h5py.File('DataVectors/gold/dbl_posteriors_TDCOSMO25_DEBIASED.h5','r') as h5:
        fpd_samps = h5['fpd_samps'][:]
        lens_param_samps = h5['lens_param_samps'][:]
        catalog_idxs = h5['catalog_idxs'][:]
        for cidx in all_cidx[30:50]:
            # find index of this cidx
            h5_idx = np.where(catalog_idxs==cidx)[0]
            # pull the fpd precision
            fpd_sigma = np.std(fpd_samps[h5_idx,:,0])
            # pull the truth fpd
            truth_row = truth_df[np.isin(truth_df['catalog_idx'],cidx)]
            # tabulate
            fpd_prec_in_order.append(fpd_sigma / np.abs(truth_row['fpd01'].item()))
            longest_td_in_order.append(np.abs(truth_row['td01'].item()))
            gamma_in_order.append(truth_row['main_deflector_parameters_gamma'].item())
            z_lens_in_order.append(truth_row['lens_light_parameters_z_source'].item())
            z_src_in_order.append(truth_row['source_parameters_z_source'].item())

    # 4MOST QUADS
    with h5py.File('DataVectors/gold/quad_posteriors_DEBIASED.h5','r') as h5:
        fpd_samps = h5['fpd_samps'][:]
        lens_param_samps = h5['lens_param_samps'][:]
        catalog_idxs = h5['catalog_idxs'][:]
        for cidx in all_cidx[50:125]:
            # find index of this cidx
            h5_idx = np.where(catalog_idxs==cidx)[0]
            # pull the fpd precision
            fpd_sigma = np.std(fpd_samps[h5_idx,:,2])
            # pull the truth fpd
            truth_row = truth_df[np.isin(truth_df['catalog_idx'],cidx)]
            # tabulate
            fpd_prec_in_order.append(fpd_sigma / np.abs(truth_row['fpd03'].item()))
            longest_td_in_order.append(np.abs(truth_row['td03'].item()))
            gamma_in_order.append(truth_row['main_deflector_parameters_gamma'].item())
            z_lens_in_order.append(truth_row['lens_light_parameters_z_source'].item())
            z_src_in_order.append(truth_row['source_parameters_z_source'].item())

    # 4MOST DOUBLES
    with h5py.File('DataVectors/gold/dbl_posteriors_DEBIASED.h5','r') as h5:
        fpd_samps = h5['fpd_samps'][:]
        lens_param_samps = h5['lens_param_samps'][:]
        catalog_idxs = h5['catalog_idxs'][:]
        for cidx in all_cidx[125:200]:
            # find index of this cidx
            h5_idx = np.where(catalog_idxs==cidx)[0]
            # pull the fpd precision
            fpd_sigma = np.std(fpd_samps[h5_idx,:,0])
            # pull the truth fpd
            truth_row = truth_df[np.isin(truth_df['catalog_idx'],cidx)]
            # tabulate
            fpd_prec_in_order.append(fpd_sigma / np.abs(truth_row['fpd01'].item()))
            longest_td_in_order.append(np.abs(truth_row['td01'].item()))
            gamma_in_order.append(truth_row['main_deflector_parameters_gamma'].item())
            z_lens_in_order.append(truth_row['lens_light_parameters_z_source'].item())
            z_src_in_order.append(truth_row['source_parameters_z_source'].item())

    # LSST Portion of Sample
    with h5py.File('DataVectors/silver/dbl_posteriors_DEBIASED.h5','r') as h5:
        dbl_fpd_samps = h5['fpd_samps'][:]
        dbl_lens_param_samps = h5['lens_param_samps'][:]
        dbl_catalog_idxs = h5['catalog_idxs'][:]
    with h5py.File('DataVectors/silver/quad_posteriors_DEBIASED.h5','r') as h5:
        quad_fpd_samps = h5['fpd_samps'][:]
        quad_lens_param_samps = h5['lens_param_samps'][:]
        quad_catalog_idxs = h5['catalog_idxs'][:]
    
    for cidx in all_cidx[200:]:
        # pull the truth fpd
        truth_row = truth_df[np.isin(truth_df['catalog_idx'],cidx)]
        # tabulate
        td03 = truth_row['td03'].item()
        # DOUBLE
        if np.isnan(td03):
            # find index of this cidx
            h5_idx = np.where(dbl_catalog_idxs==cidx)[0]
            # pull the fpd precision
            fpd_sigma = np.std(dbl_fpd_samps[h5_idx,:,0])
            fpd_prec_val = fpd_sigma / np.abs(truth_row['fpd01'].item())
            # longest td
            longest_td_val = np.abs(truth_row['td01'].item())

        # QUAD:
        else:
            # find index of this cidx
            h5_idx = np.where(quad_catalog_idxs==cidx)[0]
            # pull the fpd precision
            fpd_sigma = np.std(quad_fpd_samps[h5_idx,:,2])
            fpd_prec_val = fpd_sigma / np.abs(truth_row['fpd03'].item())
            # longest td
            longest_td_val = np.abs(truth_row['td03'].item())

        fpd_prec_in_order.append(fpd_prec_val)
        longest_td_in_order.append(longest_td_val)
        gamma_in_order.append(truth_row['main_deflector_parameters_gamma'].item())
        z_lens_in_order.append(truth_row['lens_light_parameters_z_source'].item())
        z_src_in_order.append(truth_row['source_parameters_z_source'].item())


    # now do an approximate DDT precision
    fpd_prec_in_order = np.asarray(fpd_prec_in_order)
    longest_td_in_order = np.asarray(longest_td_in_order)
    proxy_ddt_prec_in_order = np.empty(len(fpd_prec_in_order))
    proxy_ddt_prec_in_order[:40] = np.sqrt(fpd_prec_in_order[:40]**2 + 0.03**2)        
    proxy_ddt_prec_in_order[40:] = np.sqrt(fpd_prec_in_order[40:]**2 + (5/longest_td_in_order[40:])**2 )     

    # Index into truth_df by catalog_idx
    # gold_df = gold_df[~gold_df['catalog_idx'].isin(nirspec_quads_catalog_idxs)].reset_index(drop=True)
    gold_truth = truth_df[truth_df['catalog_idx'].isin(cidx_gold)].reset_index(drop=True)
    dbls_truth = truth_df[truth_df['catalog_idx'].isin(cidx_ifu_dbls)].reset_index(drop=True)
    z_lens_gold = gold_truth['lens_light_parameters_z_source'].to_numpy()
    mu_z_lens = np.mean(z_lens_gold)
    min_z_lens = np.min(z_lens_gold)
    median_z_lens = np.median(z_lens_gold)

    z_src_gold = gold_truth['source_parameters_z_source'].to_numpy()
    mu_z_src = np.mean(z_src_gold)

    from astropy.cosmology import FlatLambdaCDM
    gt_cosmo = FlatLambdaCDM(H0=70.,Om0=0.3)
    D_ds_gold = np.array(gt_cosmo.angular_diameter_distance_z1z2(z_lens_gold,z_src_gold))
    mean_D_ds = np.mean(D_ds_gold)
    Ddt_Mpc_gold = gold_truth['Ddt_Mpc'].to_numpy()

    theta_E_gold = gold_truth['main_deflector_parameters_theta_E'].to_numpy()
    mu_theta_E = np.mean(theta_E_gold)

    gamma_gold = gold_truth['main_deflector_parameters_gamma'].to_numpy()
    mu_gamma = np.mean(gamma_gold)
    median_gamma = np.median(gamma_gold)

    z_src_dbls = np.abs(dbls_truth['source_parameters_z_source'].to_numpy())
    z_lens_dbls = np.abs(dbls_truth['lens_light_parameters_z_source'].to_numpy())
    gamma_dbls = dbls_truth['main_deflector_parameters_gamma'].to_numpy()
    dbls_mu_gamma = np.mean(gamma_dbls)

    m_app_ps = gold_truth['point_source_parameters_mag_app'].to_numpy()
    min_pointsource_magapp = np.min(m_app_ps)

    beta_ani_gold = gold_truth['beta_ani'].to_numpy()
    lambda_int_gold = gold_truth['lambda_int'].to_numpy()

    td01 = gold_truth['td01'].to_numpy()
    td02 = gold_truth['td02'].to_numpy()
    td03 = gold_truth['td03'].to_numpy()
    shortest_td_gold = np.abs(td01)
    longest_td = np.abs(td03)
    longest_td[np.isnan(longest_td)] = np.abs(td01[np.isnan(longest_td)])
    mean_longest_td = np.mean(longest_td)
    mean_longest_td_prec = np.mean(100 * 5/longest_td)
    max_longest_td = np.max(longest_td)
    td_longer_than_200 = np.sum(longest_td>200)


    fpd01 = gold_truth['fpd01'].to_numpy()
    fpd02 = gold_truth['fpd02'].to_numpy()
    fpd03 = gold_truth['fpd03'].to_numpy()
    longest_fpd = np.abs(fpd03)
    longest_fpd[np.isnan(longest_fpd)] = np.abs(fpd01[np.isnan(longest_fpd)])
    mean_longest_fpd = np.mean(longest_fpd)

    # lens light properties
    m_app_lens_gold = gold_truth['lens_light_parameters_mag_app'].to_numpy()
    R_sersic_lens_gold = gold_truth['lens_light_parameters_R_sersic'].to_numpy()
    n_sersic_lens_gold = gold_truth['lens_light_parameters_n_sersic'].to_numpy()
    

    test_chain = curr_chain[:,burnin:,:].reshape((-1,curr_chain.shape[2]))
    #sigma_h0 = np.std(test_chain[:,0],ddof=1)
    arviz_hdi = az.hdi(test_chain[:,0], hdi_prob=.68)
    sigma_h0 = (arviz_hdi[1] - arviz_hdi[0])/2
    arviz_hdi = az.hdi(test_chain[:,4], hdi_prob=.68)
    sigma_mu_lint = (arviz_hdi[1] - arviz_hdi[0])/2
    arviz_hdi = az.hdi(test_chain[:,1], hdi_prob=.68)
    sigma_omegaM = (arviz_hdi[1] - arviz_hdi[0])/2
    print('seed %d'%(j))
    print('sigma h0: ', sigma_h0)
    print('sigma omegaM: ', sigma_omegaM)
    zp,de_fom = DE_fom(curr_chain,burnin)
    de_fom_list.append(de_fom)


    lsst_proxy_ddt = proxy_ddt_prec_in_order[200:]
    #num_special = np.sum(lsst_proxy_ddt>0.6)
    num_special = np.sum((z_lens_gold<0.5) & (longest_td > 100.))
    #np.sum((gamma_gold > 2.1) & (longest_td > 50.))
    #num_special = np.sum((gamma_gold > 2.25))

    chosen_param = num_special
    current_param_list.append(chosen_param)
    # TODO: define the bins
    #hist_bins = [0.,0.05,0.1,0.15,0.2,0.25,0.3,0.35,0.4,0.45,0.5,0.55,0.6]
    #hist_bins = [0.02,0.025,0.03,0.035,0.04,0.045,0.05,0.055,0.06,0.065,0.07,0.075,0.08,0.085]#,0.1,0.15,0.2,0.25,0.3,0.35,0.4,0.45,0.5,0.55,0.6]
    hist_bins = np.arange(0.2,2.2,step=0.2)
    #plt.hist(R_sersic_lens_gold,label='seed %d'%(seed_nums[j]),histtype='step',bins=hist_bins,color=scatter_colors[j])
    z_lens_in_order = np.asarray(z_lens_in_order)
    z_src_in_order = np.asarray(z_src_in_order)
    plt.scatter(z_lens_gold,z_src_gold,label='seed %d'%(j),s=80,c=scatter_colors[j],marker=scatter_marker[j],zorder=100*(5-j))
    #if j == 5:
    #    plt.scatter(chosen_param,de_fom,label='seed %d'%(j),s=180,marker='*',edgecolors='peru')
    #else:
    #    plt.scatter(chosen_param,de_fom,label='seed %d'%(j),s=90)

corr_of_param = np.corrcoef(current_param_list,de_fom_list)
#print(corr_of_param)

#plt.ylabel('DE FOM [$\sigma(w_0) \sigma(w_p)$]$^{-1}$',fontsize=15)
plt.xlabel(r'z_lens',fontsize=17)
plt.ylabel(r'z_src',fontsize=17)
#plt.legend(fontsize=12)
plt.title('JWST Quads, Pantheon+ $\Omega_m$ prior')#, corr=%.2f'%(corr_of_param[0,1]))
#plt.ylim([2.,22.])
plt.savefig('/Users/smericks/Desktop/baseline_fluctuation_precision_0Mprior.pdf')

In [ ]:
import arviz as az
burnin=20000
chains = [
    #np.transpose(chains_dict['exp1_3']['w0wa_seed1_OmegaM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp1_3']['w0wa_seed2_OmegaM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp1_3']['w0wa_seed3_OmegaM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp1_3']['w0wa_seed4_OmegaM']['chain'], axes=(1,0,2)),
    #np.transpose(chains_dict['exp1_3']['w0wa_seed5_OmegaM']['chain'], axes=(1,0,2)),
    #np.transpose(chains_dict['exp1_3']['w0wa_seed6_OmegaM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp1_3']['w0wa_seed7_OmegaM']['chain'], axes=(1,0,2)),
    #np.transpose(chains_dict['exp1_3']['w0wa_seed8_OmegaM']['chain'], axes=(1,0,2)),
    #np.transpose(chains_dict['exp1_3']['w0wa_seed9_OmegaM']['chain'], axes=(1,0,2)),
    #np.transpose(chains_dict['exp1_3']['w0wa_seed10_OmegaM']['chain'], axes=(1,0,2))
]
truth_df = pd.read_csv('DataVectors/gold/truth_metadata.csv')
scatter_colors = ['C1','C2','C3','C6']
scatter_marker = ["1","2","3","4","o"]
seed_nums = [1,2,3,6]

de_fom_list = []
current_param_list = []
plt.figure(dpi=200)
print('len chains',len(chains))
for j,curr_chain in enumerate(chains):

    print('seed %d'%(seed_nums[j]))
    all_cidx = np.load('InferenceRuns/exp0_2/catalog_idxs_seed%d.npy'%(seed_nums[j]))
    print('jwst cidx: ', all_cidx[:10])

    cidx_gold = all_cidx[30:50]

    # INDEXING KEEPING ORDERING THE EXACT SAME (for things that aren't means!)
    fpd_prec_in_order = []
    longest_td_in_order = []
    z_lens_in_order = []
    z_src_in_order = []
    gamma_in_order = []
    # MUSE QUADS
    with h5py.File('DataVectors/gold/quad_posteriors_TDCOSMO25_DEBIASED.h5','r') as h5:
        fpd_samps = h5['fpd_samps'][:]
        lens_param_samps = h5['lens_param_samps'][:]
        catalog_idxs = h5['catalog_idxs'][:]
        for cidx in all_cidx[10:30]:
            # find index of this cidx
            h5_idx = np.where(catalog_idxs==cidx)[0]
            # pull the fpd precision
            fpd_sigma = np.std(fpd_samps[h5_idx,:,2])
            # pull the truth fpd
            truth_row = truth_df[np.isin(truth_df['catalog_idx'],cidx)]
            # tabulate
            fpd_prec_in_order.append(fpd_sigma / np.abs(truth_row['fpd03'].item()))
            longest_td_in_order.append(np.abs(truth_row['td03'].item()))
            gamma_in_order.append(truth_row['main_deflector_parameters_gamma'].item())
            z_lens_in_order.append(truth_row['lens_light_parameters_z_source'].item())
            z_src_in_order.append(truth_row['source_parameters_z_source'].item())

    # MUSE DOUBLES
    with h5py.File('DataVectors/gold/dbl_posteriors_TDCOSMO25_DEBIASED.h5','r') as h5:
        fpd_samps = h5['fpd_samps'][:]
        lens_param_samps = h5['lens_param_samps'][:]
        catalog_idxs = h5['catalog_idxs'][:]
        for cidx in all_cidx[30:50]:
            # find index of this cidx
            h5_idx = np.where(catalog_idxs==cidx)[0]
            # pull the fpd precision
            fpd_sigma = np.std(fpd_samps[h5_idx,:,0])
            # pull the truth fpd
            truth_row = truth_df[np.isin(truth_df['catalog_idx'],cidx)]
            # tabulate
            fpd_prec_in_order.append(fpd_sigma / np.abs(truth_row['fpd01'].item()))
            longest_td_in_order.append(np.abs(truth_row['td01'].item()))
            gamma_in_order.append(truth_row['main_deflector_parameters_gamma'].item())
            z_lens_in_order.append(truth_row['lens_light_parameters_z_source'].item())
            z_src_in_order.append(truth_row['source_parameters_z_source'].item())

    # 4MOST QUADS
    with h5py.File('DataVectors/gold/quad_posteriors_DEBIASED.h5','r') as h5:
        fpd_samps = h5['fpd_samps'][:]
        lens_param_samps = h5['lens_param_samps'][:]
        catalog_idxs = h5['catalog_idxs'][:]
        for cidx in all_cidx[50:125]:
            # find index of this cidx
            h5_idx = np.where(catalog_idxs==cidx)[0]
            # pull the fpd precision
            fpd_sigma = np.std(fpd_samps[h5_idx,:,2])
            # pull the truth fpd
            truth_row = truth_df[np.isin(truth_df['catalog_idx'],cidx)]
            # tabulate
            fpd_prec_in_order.append(fpd_sigma / np.abs(truth_row['fpd03'].item()))
            longest_td_in_order.append(np.abs(truth_row['td03'].item()))
            gamma_in_order.append(truth_row['main_deflector_parameters_gamma'].item())
            z_lens_in_order.append(truth_row['lens_light_parameters_z_source'].item())
            z_src_in_order.append(truth_row['source_parameters_z_source'].item())

    # 4MOST DOUBLES
    with h5py.File('DataVectors/gold/dbl_posteriors_DEBIASED.h5','r') as h5:
        fpd_samps = h5['fpd_samps'][:]
        lens_param_samps = h5['lens_param_samps'][:]
        catalog_idxs = h5['catalog_idxs'][:]
        for cidx in all_cidx[125:200]:
            # find index of this cidx
            h5_idx = np.where(catalog_idxs==cidx)[0]
            # pull the fpd precision
            fpd_sigma = np.std(fpd_samps[h5_idx,:,0])
            # pull the truth fpd
            truth_row = truth_df[np.isin(truth_df['catalog_idx'],cidx)]
            # tabulate
            fpd_prec_in_order.append(fpd_sigma / np.abs(truth_row['fpd01'].item()))
            longest_td_in_order.append(np.abs(truth_row['td01'].item()))
            gamma_in_order.append(truth_row['main_deflector_parameters_gamma'].item())
            z_lens_in_order.append(truth_row['lens_light_parameters_z_source'].item())
            z_src_in_order.append(truth_row['source_parameters_z_source'].item())

    # LSST Portion of Sample
    with h5py.File('DataVectors/silver/dbl_posteriors_DEBIASED.h5','r') as h5:
        dbl_fpd_samps = h5['fpd_samps'][:]
        dbl_lens_param_samps = h5['lens_param_samps'][:]
        dbl_catalog_idxs = h5['catalog_idxs'][:]
    with h5py.File('DataVectors/silver/quad_posteriors_DEBIASED.h5','r') as h5:
        quad_fpd_samps = h5['fpd_samps'][:]
        quad_lens_param_samps = h5['lens_param_samps'][:]
        quad_catalog_idxs = h5['catalog_idxs'][:]
    
    for cidx in all_cidx[200:]:
        # pull the truth fpd
        truth_row = truth_df[np.isin(truth_df['catalog_idx'],cidx)]
        # tabulate
        td03 = truth_row['td03'].item()
        # DOUBLE
        if np.isnan(td03):
            # find index of this cidx
            h5_idx = np.where(dbl_catalog_idxs==cidx)[0]
            # pull the fpd precision
            fpd_sigma = np.std(dbl_fpd_samps[h5_idx,:,0])
            fpd_prec_val = fpd_sigma / np.abs(truth_row['fpd01'].item())
            # longest td
            longest_td_val = np.abs(truth_row['td01'].item())

        # QUAD:
        else:
            # find index of this cidx
            h5_idx = np.where(quad_catalog_idxs==cidx)[0]
            # pull the fpd precision
            fpd_sigma = np.std(quad_fpd_samps[h5_idx,:,2])
            fpd_prec_val = fpd_sigma / np.abs(truth_row['fpd03'].item())
            # longest td
            longest_td_val = np.abs(truth_row['td03'].item())

        fpd_prec_in_order.append(fpd_prec_val)
        longest_td_in_order.append(longest_td_val)
        gamma_in_order.append(truth_row['main_deflector_parameters_gamma'].item())
        z_lens_in_order.append(truth_row['lens_light_parameters_z_source'].item())
        z_src_in_order.append(truth_row['source_parameters_z_source'].item())


    # now do an approximate DDT precision
    fpd_prec_in_order = np.asarray(fpd_prec_in_order)
    longest_td_in_order = np.asarray(longest_td_in_order)
    proxy_ddt_prec_in_order = np.empty(len(fpd_prec_in_order))
    proxy_ddt_prec_in_order[:40] = np.sqrt(fpd_prec_in_order[:40]**2 + 0.03**2)        
    proxy_ddt_prec_in_order[40:] = np.sqrt(fpd_prec_in_order[40:]**2 + (5/longest_td_in_order[40:])**2 )     

    # Index into truth_df by catalog_idx
    # gold_df = gold_df[~gold_df['catalog_idx'].isin(nirspec_quads_catalog_idxs)].reset_index(drop=True)
    gold_truth = truth_df[truth_df['catalog_idx'].isin(cidx_gold)].reset_index(drop=True)
    dbls_truth = truth_df[truth_df['catalog_idx'].isin(cidx_ifu_dbls)].reset_index(drop=True)
    z_lens_gold = gold_truth['lens_light_parameters_z_source'].to_numpy()
    mu_z_lens = np.mean(z_lens_gold)
    min_z_lens = np.min(z_lens_gold)
    median_z_lens = np.median(z_lens_gold)

    z_src_gold = gold_truth['source_parameters_z_source'].to_numpy()
    mu_z_src = np.mean(z_src_gold)

    from astropy.cosmology import FlatLambdaCDM
    gt_cosmo = FlatLambdaCDM(H0=70.,Om0=0.3)
    D_ds_gold = np.array(gt_cosmo.angular_diameter_distance_z1z2(z_lens_gold,z_src_gold))
    mean_D_ds = np.mean(D_ds_gold)
    Ddt_Mpc_gold = gold_truth['Ddt_Mpc'].to_numpy()

    theta_E_gold = gold_truth['main_deflector_parameters_theta_E'].to_numpy()
    mu_theta_E = np.mean(theta_E_gold)

    gamma_gold = gold_truth['main_deflector_parameters_gamma'].to_numpy()
    mu_gamma = np.mean(gamma_gold)
    median_gamma = np.median(gamma_gold)

    m_app_ps = gold_truth['point_source_parameters_mag_app'].to_numpy()
    min_pointsource_magapp = np.min(m_app_ps)

    beta_ani_gold = gold_truth['beta_ani'].to_numpy()
    lambda_int_gold = gold_truth['lambda_int'].to_numpy()

    td01 = gold_truth['td01'].to_numpy()
    td02 = gold_truth['td02'].to_numpy()
    td03 = gold_truth['td03'].to_numpy()
    shortest_td_gold = np.abs(td01)
    longest_td = np.abs(td03)
    longest_td[np.isnan(longest_td)] = np.abs(td01[np.isnan(longest_td)])
    mean_longest_td = np.mean(longest_td)
    mean_longest_td_prec = np.mean(100 * 5/longest_td)
    max_longest_td = np.max(longest_td)
    td_longer_than_200 = np.sum(longest_td>200)


    fpd01 = gold_truth['fpd01'].to_numpy()
    fpd02 = gold_truth['fpd02'].to_numpy()
    fpd03 = gold_truth['fpd03'].to_numpy()
    longest_fpd = np.abs(fpd03)
    longest_fpd[np.isnan(longest_fpd)] = np.abs(fpd01[np.isnan(longest_fpd)])
    mean_longest_fpd = np.mean(longest_fpd)

    # lens light properties
    m_app_lens_gold = gold_truth['lens_light_parameters_mag_app'].to_numpy()
    R_sersic_lens_gold = gold_truth['lens_light_parameters_R_sersic'].to_numpy()
    n_sersic_lens_gold = gold_truth['lens_light_parameters_n_sersic'].to_numpy()
    

    test_chain = curr_chain[:,burnin:,:].reshape((-1,curr_chain.shape[2]))
    #sigma_h0 = np.std(test_chain[:,0],ddof=1)
    arviz_hdi = az.hdi(test_chain[:,0], hdi_prob=.68)
    sigma_h0 = (arviz_hdi[1] - arviz_hdi[0])/2
    arviz_hdi = az.hdi(test_chain[:,4], hdi_prob=.68)
    sigma_mu_lint = (arviz_hdi[1] - arviz_hdi[0])/2
    arviz_hdi = az.hdi(test_chain[:,1], hdi_prob=.68)
    sigma_omegaM = (arviz_hdi[1] - arviz_hdi[0])/2
    print('seed %d'%(j))
    print('sigma h0: ', sigma_h0)
    print('sigma omegaM: ', sigma_omegaM)
    zp,de_fom = DE_fom(curr_chain,burnin)
    de_fom_list.append(de_fom)


    lsst_proxy_ddt = proxy_ddt_prec_in_order[200:]
    #num_special = np.sum(lsst_proxy_ddt>0.6)
    num_special = np.sum((z_lens_gold<0.5) & (longest_td > 100.))
    #np.sum((gamma_gold > 2.1) & (longest_td > 50.))
    #num_special = np.sum((gamma_gold > 2.25))

    chosen_param = num_special
    current_param_list.append(chosen_param)
    # TODO: define the bins
    #hist_bins = [0.,0.05,0.1,0.15,0.2,0.25,0.3,0.35,0.4,0.45,0.5,0.55,0.6]
    #hist_bins = [0.02,0.025,0.03,0.035,0.04,0.045,0.05,0.055,0.06,0.065,0.07,0.075,0.08,0.085]#,0.1,0.15,0.2,0.25,0.3,0.35,0.4,0.45,0.5,0.55,0.6]
    hist_bins = np.arange(0.2,2.2,step=0.2)
    #plt.hist(R_sersic_lens_gold,label='seed %d'%(seed_nums[j]),histtype='step',bins=hist_bins,color=scatter_colors[j])
    z_lens_in_order = np.asarray(z_lens_in_order)
    plt.scatter(z_lens_gold,z_src_gold,label='seed %d'%(j),s=80,c=scatter_colors[j],marker=scatter_marker[j],zorder=100*(5-j))
    #if j == 5:
    #    plt.scatter(chosen_param,de_fom,label='seed %d'%(j),s=180,marker='*',edgecolors='peru')
    #else:
    #    plt.scatter(chosen_param,de_fom,label='seed %d'%(j),s=90)

corr_of_param = np.corrcoef(current_param_list,de_fom_list)
#print(corr_of_param)

#plt.ylabel('DE FOM [$\sigma(w_0) \sigma(w_p)$]$^{-1}$',fontsize=15)
plt.xlabel(r'z_lens',fontsize=17)
plt.ylabel(r'z_src',fontsize=17)
#plt.legend(fontsize=12)
plt.title('JWST Quads, Pantheon+ $\Omega_m$ prior')#, corr=%.2f'%(corr_of_param[0,1]))
#plt.ylim([2.,22.])

In [ ]:
import arviz as az
burnin=20000
chains = [
    np.transpose(chains_dict['exp0_2']['w0wa_seed1_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed2_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed3_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed4_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed5_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed6_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed7_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed8_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed9_pantheonOM']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed10_pantheonOM']['chain'], axes=(1,0,2))
]
truth_df = pd.read_csv('DataVectors/gold/truth_metadata.csv')
scatter_colors = ['C1','C2','C3','C6']
scatter_marker = ["1","2","3","4","o"]
seed_nums = [0,1,2,3,4,5,6,7,8,9]

de_fom_list = []
current_param_list = []
plt.figure(dpi=200)
for j,curr_chain in enumerate(chains):

    print('seed %d'%(seed_nums[j]))
    all_cidx = np.load('InferenceRuns/exp0_2/catalog_idxs_seed%d.npy'%(seed_nums[j]))

    cidx_gold = all_cidx[:800]
    cidx_ifu_dbls = all_cidx[30:50]

    # INDEXING KEEPING ORDERING THE EXACT SAME (for things that aren't means!)
    fpd_prec_in_order = []
    longest_td_in_order = []
    z_lens_in_order = []
    z_src_in_order = []
    gamma_in_order = []
    # MUSE QUADS
    with h5py.File('DataVectors/gold/quad_posteriors_TDCOSMO25_DEBIASED.h5','r') as h5:
        fpd_samps = h5['fpd_samps'][:]
        lens_param_samps = h5['lens_param_samps'][:]
        catalog_idxs = h5['catalog_idxs'][:]
        for cidx in all_cidx[10:30]:
            # find index of this cidx
            h5_idx = np.where(catalog_idxs==cidx)[0]
            # pull the fpd precision
            fpd_sigma = np.std(fpd_samps[h5_idx,:,2])
            # pull the truth fpd
            truth_row = truth_df[np.isin(truth_df['catalog_idx'],cidx)]
            # tabulate
            fpd_prec_in_order.append(fpd_sigma / np.abs(truth_row['fpd03'].item()))
            longest_td_in_order.append(np.abs(truth_row['td03'].item()))
            gamma_in_order.append(truth_row['main_deflector_parameters_gamma'].item())
            z_lens_in_order.append(truth_row['lens_light_parameters_z_source'].item())
            z_src_in_order.append(truth_row['source_parameters_z_source'].item())

    # MUSE DOUBLES
    with h5py.File('DataVectors/gold/dbl_posteriors_TDCOSMO25_DEBIASED.h5','r') as h5:
        fpd_samps = h5['fpd_samps'][:]
        lens_param_samps = h5['lens_param_samps'][:]
        catalog_idxs = h5['catalog_idxs'][:]
        for cidx in all_cidx[30:50]:
            # find index of this cidx
            h5_idx = np.where(catalog_idxs==cidx)[0]
            # pull the fpd precision
            fpd_sigma = np.std(fpd_samps[h5_idx,:,0])
            # pull the truth fpd
            truth_row = truth_df[np.isin(truth_df['catalog_idx'],cidx)]
            # tabulate
            fpd_prec_in_order.append(fpd_sigma / np.abs(truth_row['fpd01'].item()))
            longest_td_in_order.append(np.abs(truth_row['td01'].item()))
            gamma_in_order.append(truth_row['main_deflector_parameters_gamma'].item())
            z_lens_in_order.append(truth_row['lens_light_parameters_z_source'].item())
            z_src_in_order.append(truth_row['source_parameters_z_source'].item())

    # 4MOST QUADS
    with h5py.File('DataVectors/gold/quad_posteriors_DEBIASED.h5','r') as h5:
        fpd_samps = h5['fpd_samps'][:]
        lens_param_samps = h5['lens_param_samps'][:]
        catalog_idxs = h5['catalog_idxs'][:]
        for cidx in all_cidx[50:125]:
            # find index of this cidx
            h5_idx = np.where(catalog_idxs==cidx)[0]
            # pull the fpd precision
            fpd_sigma = np.std(fpd_samps[h5_idx,:,2])
            # pull the truth fpd
            truth_row = truth_df[np.isin(truth_df['catalog_idx'],cidx)]
            # tabulate
            fpd_prec_in_order.append(fpd_sigma / np.abs(truth_row['fpd03'].item()))
            longest_td_in_order.append(np.abs(truth_row['td03'].item()))
            gamma_in_order.append(truth_row['main_deflector_parameters_gamma'].item())
            z_lens_in_order.append(truth_row['lens_light_parameters_z_source'].item())
            z_src_in_order.append(truth_row['source_parameters_z_source'].item())

    # 4MOST DOUBLES
    with h5py.File('DataVectors/gold/dbl_posteriors_DEBIASED.h5','r') as h5:
        fpd_samps = h5['fpd_samps'][:]
        lens_param_samps = h5['lens_param_samps'][:]
        catalog_idxs = h5['catalog_idxs'][:]
        for cidx in all_cidx[125:200]:
            # find index of this cidx
            h5_idx = np.where(catalog_idxs==cidx)[0]
            # pull the fpd precision
            fpd_sigma = np.std(fpd_samps[h5_idx,:,0])
            # pull the truth fpd
            truth_row = truth_df[np.isin(truth_df['catalog_idx'],cidx)]
            # tabulate
            fpd_prec_in_order.append(fpd_sigma / np.abs(truth_row['fpd01'].item()))
            longest_td_in_order.append(np.abs(truth_row['td01'].item()))
            gamma_in_order.append(truth_row['main_deflector_parameters_gamma'].item())
            z_lens_in_order.append(truth_row['lens_light_parameters_z_source'].item())
            z_src_in_order.append(truth_row['source_parameters_z_source'].item())

    # LSST Portion of Sample
    with h5py.File('DataVectors/silver/dbl_posteriors_DEBIASED.h5','r') as h5:
        dbl_fpd_samps = h5['fpd_samps'][:]
        dbl_lens_param_samps = h5['lens_param_samps'][:]
        dbl_catalog_idxs = h5['catalog_idxs'][:]
    with h5py.File('DataVectors/silver/quad_posteriors_DEBIASED.h5','r') as h5:
        quad_fpd_samps = h5['fpd_samps'][:]
        quad_lens_param_samps = h5['lens_param_samps'][:]
        quad_catalog_idxs = h5['catalog_idxs'][:]
    
    for cidx in all_cidx[200:]:
        # pull the truth fpd
        truth_row = truth_df[np.isin(truth_df['catalog_idx'],cidx)]
        # tabulate
        td03 = truth_row['td03'].item()
        # DOUBLE
        if np.isnan(td03):
            # find index of this cidx
            h5_idx = np.where(dbl_catalog_idxs==cidx)[0]
            # pull the fpd precision
            fpd_sigma = np.std(dbl_fpd_samps[h5_idx,:,0])
            fpd_prec_val = fpd_sigma / np.abs(truth_row['fpd01'].item())
            # longest td
            longest_td_val = np.abs(truth_row['td01'].item())

        # QUAD:
        else:
            # find index of this cidx
            h5_idx = np.where(quad_catalog_idxs==cidx)[0]
            # pull the fpd precision
            fpd_sigma = np.std(quad_fpd_samps[h5_idx,:,2])
            fpd_prec_val = fpd_sigma / np.abs(truth_row['fpd03'].item())
            # longest td
            longest_td_val = np.abs(truth_row['td03'].item())

        fpd_prec_in_order.append(fpd_prec_val)
        longest_td_in_order.append(longest_td_val)
        gamma_in_order.append(truth_row['main_deflector_parameters_gamma'].item())
        z_lens_in_order.append(truth_row['lens_light_parameters_z_source'].item())
        z_src_in_order.append(truth_row['source_parameters_z_source'].item())


    # now do an approximate DDT precision
    fpd_prec_in_order = np.asarray(fpd_prec_in_order)
    longest_td_in_order = np.asarray(longest_td_in_order)
    proxy_ddt_prec_in_order = np.empty(len(fpd_prec_in_order))
    proxy_ddt_prec_in_order[:40] = np.sqrt(fpd_prec_in_order[:40]**2 + 0.03**2)        
    proxy_ddt_prec_in_order[40:] = np.sqrt(fpd_prec_in_order[40:]**2 + (5/longest_td_in_order[40:])**2 )     

    # Index into truth_df by catalog_idx
    # gold_df = gold_df[~gold_df['catalog_idx'].isin(nirspec_quads_catalog_idxs)].reset_index(drop=True)
    gold_truth = truth_df[truth_df['catalog_idx'].isin(cidx_gold)].reset_index(drop=True)
    dbls_truth = truth_df[truth_df['catalog_idx'].isin(cidx_ifu_dbls)].reset_index(drop=True)
    z_lens_gold = gold_truth['lens_light_parameters_z_source'].to_numpy()
    mu_z_lens = np.mean(z_lens_gold)
    min_z_lens = np.min(z_lens_gold)
    median_z_lens = np.median(z_lens_gold)

    z_src_gold = gold_truth['source_parameters_z_source'].to_numpy()
    mu_z_src = np.mean(z_src_gold)

    from astropy.cosmology import FlatLambdaCDM
    gt_cosmo = FlatLambdaCDM(H0=70.,Om0=0.3)
    D_ds_gold = np.array(gt_cosmo.angular_diameter_distance_z1z2(z_lens_gold,z_src_gold))
    mean_D_ds = np.mean(D_ds_gold)
    Ddt_Mpc_gold = gold_truth['Ddt_Mpc'].to_numpy()

    theta_E_gold = gold_truth['main_deflector_parameters_theta_E'].to_numpy()
    mu_theta_E = np.mean(theta_E_gold)

    gamma_gold = gold_truth['main_deflector_parameters_gamma'].to_numpy()
    mu_gamma = np.mean(gamma_gold)
    median_gamma = np.median(gamma_gold)

    z_src_dbls = np.abs(dbls_truth['source_parameters_z_source'].to_numpy())
    z_lens_dbls = np.abs(dbls_truth['lens_light_parameters_z_source'].to_numpy())
    gamma_dbls = dbls_truth['main_deflector_parameters_gamma'].to_numpy()
    dbls_mu_gamma = np.mean(gamma_dbls)

    m_app_ps = gold_truth['point_source_parameters_mag_app'].to_numpy()
    min_pointsource_magapp = np.min(m_app_ps)

    beta_ani_gold = gold_truth['beta_ani'].to_numpy()
    lambda_int_gold = gold_truth['lambda_int'].to_numpy()

    td01 = gold_truth['td01'].to_numpy()
    td02 = gold_truth['td02'].to_numpy()
    td03 = gold_truth['td03'].to_numpy()
    shortest_td_gold = np.abs(td01)
    longest_td = np.abs(td03)
    longest_td[np.isnan(longest_td)] = np.abs(td01[np.isnan(longest_td)])
    mean_longest_td = np.mean(longest_td)
    mean_longest_td_prec = np.mean(100 * 5/longest_td)
    max_longest_td = np.max(longest_td)
    td_longer_than_200 = np.sum(longest_td>200)


    fpd01 = gold_truth['fpd01'].to_numpy()
    fpd02 = gold_truth['fpd02'].to_numpy()
    fpd03 = gold_truth['fpd03'].to_numpy()
    longest_fpd = np.abs(fpd03)
    longest_fpd[np.isnan(longest_fpd)] = np.abs(fpd01[np.isnan(longest_fpd)])
    mean_longest_fpd = np.mean(longest_fpd)

    # lens light properties
    m_app_lens_gold = gold_truth['lens_light_parameters_mag_app'].to_numpy()
    R_sersic_lens_gold = gold_truth['lens_light_parameters_R_sersic'].to_numpy()
    n_sersic_lens_gold = gold_truth['lens_light_parameters_n_sersic'].to_numpy()
    

    test_chain = curr_chain[:,burnin:,:].reshape((-1,curr_chain.shape[2]))
    #sigma_h0 = np.std(test_chain[:,0],ddof=1)
    arviz_hdi = az.hdi(test_chain[:,0], hdi_prob=.68)
    sigma_h0 = (arviz_hdi[1] - arviz_hdi[0])/2
    arviz_hdi = az.hdi(test_chain[:,4], hdi_prob=.68)
    sigma_mu_lint = (arviz_hdi[1] - arviz_hdi[0])/2
    arviz_hdi = az.hdi(test_chain[:,1], hdi_prob=.68)
    sigma_omegaM = (arviz_hdi[1] - arviz_hdi[0])/2
    print('seed %d'%(j))
    print('sigma h0: ', sigma_h0)
    print('sigma omegaM: ', sigma_omegaM)
    zp,de_fom = DE_fom(curr_chain,burnin)
    de_fom_list.append(de_fom)


    z_lens_in_order = np.asarray(z_lens_in_order)
    z_src_in_order = np.asarray(z_src_in_order)
    lsst_proxy_ddt = proxy_ddt_prec_in_order[200:]
    #num_special = np.sum(lsst_proxy_ddt>0.6)
    num_special = np.sum((proxy_ddt_prec_in_order<0.1) & (z_src_in_order>2.))
    #np.sum((gamma_gold > 2.1) & (longest_td > 50.))
    #num_special = np.sum((gamma_gold > 2.25))

    chosen_param = num_special
    current_param_list.append(chosen_param)
    # TODO: define the bins
    #hist_bins = [0.,0.05,0.1,0.15,0.2,0.25,0.3,0.35,0.4,0.45,0.5,0.55,0.6]
    #hist_bins = [0.02,0.025,0.03,0.035,0.04,0.045,0.05,0.055,0.06,0.065,0.07,0.075,0.08,0.085]#,0.1,0.15,0.2,0.25,0.3,0.35,0.4,0.45,0.5,0.55,0.6]
    hist_bins = np.arange(0.2,2.2,step=0.2)
    #plt.hist(R_sersic_lens_gold,label='seed %d'%(seed_nums[j]),histtype='step',bins=hist_bins,color=scatter_colors[j])
    #plt.scatter(z_lens_gold,longest_td,label='seed %d'%(j),s=80,c=scatter_colors[j],marker=scatter_marker[j],zorder=100*(5-j))
    if j == 5:
        plt.scatter(chosen_param,de_fom,label='seed %d'%(j),s=180,marker='*',edgecolors='peru')
    else:
        plt.scatter(chosen_param,de_fom,label='seed %d'%(j),s=90)

corr_of_param = np.corrcoef(current_param_list,de_fom_list)
#print(corr_of_param)

plt.ylabel('DE FOM [$\sigma(w_0) \sigma(w_p)$]$^{-1}$',fontsize=15)
plt.xlabel(r'num DDT prec < 0.1 AND z_src>2.',fontsize=17)
#plt.ylabel(r'time-delay',fontsize=17)
#plt.legend(fontsize=12)
plt.title('Whole Sample, Pantheon+ $\Omega_m$ prior')#, corr=%.2f'%(corr_of_param[0,1]))
#plt.ylim([2.,22.])
#plt.savefig('/Users/smericks/Desktop/baseline_fluctuation_precision_0Mprior.pdf')

In [ ]:
from astropy.visualization import simple_norm

jwst_cands = [  51 ,  63 , 110 , 159 , 176 , 196 , 257  ,448 , 513 , 559 , 624 , 732 , 798  ,938,
 1006 ,1007 ,1012 ,1058 ,1118 ,1146, 1169, 1189, 1245, 1284, 1287, 1302, 1378 ,1391,
 1402 ,1445 ,1515 ,1526]

print('%d lenses avail'%(len(jwst_cands)))

with h5py.File('DataVectors/gold/image_models.h5','r') as h5:
    print(h5.keys())
    images_cidxs = h5['catalog_idx'][:]
    gold_images = h5['images_array'][:]

# select random 600 images
fig,axs = plt.subplots(4,8,figsize=(20,10))
counter = 0
for i in range(0,4):
    for j in range(0,8):
        chosen_idx = np.where(images_cidxs == jwst_cands[counter])
        if counter == 0:
            norm = simple_norm(gold_images[chosen_idx][0],stretch='log',min_cut=1e-6)
        axs[i,j].imshow(gold_images[chosen_idx][0],norm=norm)
        axs[i,j].axis('off')
        axs[i,j].text(20,20,jwst_cands[counter],color='white')
        counter+=1

In [ ]:
proxy_ddt_prec_in_order

In [ ]:
plt.hist(proxy_ddt_prec_in_order)
np.max(proxy_ddt_prec_in_order)

In [ ]:
for key in truth_df.keys():
    print(key)

In [ ]:
import arviz as az
burnin=20000
chains = [
    np.transpose(chains_dict['exp0_2']['w0wa_seed1_pantheonOM']['chain'][:,:50000], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed2_pantheonOM']['chain'][:,:50000], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed3_pantheonOM']['chain'][:,:50000], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed4_pantheonOM']['chain'][:,:50000], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed5_pantheonOM']['chain'][:,:50000], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed6_pantheonOM']['chain'][:,:50000], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed7_pantheonOM']['chain'][:,:50000], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed8_pantheonOM']['chain'][:,:50000], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed9_pantheonOM']['chain'][:,:50000], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed10_pantheonOM']['chain'][:,:50000], axes=(1,0,2))
]
truth_df = pd.read_csv('DataVectors/gold/truth_metadata.csv')


de_fom_list = []
current_param_list = []
# Define colormap and normalization
from matplotlib.colors import Normalize
defom_norm = Normalize(vmin=7., vmax=18.)

# Scatter plot
plt.figure(dpi=200)
for j,curr_chain in enumerate(chains):


    cidx = np.load('InferenceRuns/exp0_2/catalog_idxs_seed%d.npy'%(j))

    cidx_gold = cidx[:50]
    cidx_ifu_dbls = cidx[30:50]

    # Index into truth_df by catalog_idx
    # gold_df = gold_df[~gold_df['catalog_idx'].isin(nirspec_quads_catalog_idxs)].reset_index(drop=True)
    gold_truth = truth_df[truth_df['catalog_idx'].isin(cidx_gold)].reset_index(drop=True)
    dbls_truth = truth_df[truth_df['catalog_idx'].isin(cidx_ifu_dbls)].reset_index(drop=True)
    z_lens_gold = gold_truth['lens_light_parameters_z_source'].to_numpy()
    mu_z_lens = np.mean(z_lens_gold)
    min_z_lens = np.min(z_lens_gold)
    median_z_lens = np.median(z_lens_gold)

    z_src_gold = gold_truth['source_parameters_z_source'].to_numpy()
    mu_z_src = np.mean(z_src_gold)
    max_z_src = np.max(z_src_gold)

    from astropy.cosmology import FlatLambdaCDM
    gt_cosmo = FlatLambdaCDM(H0=70.,Om0=0.3)
    D_ds_gold = np.array(gt_cosmo.angular_diameter_distance_z1z2(z_lens_gold,z_src_gold))
    mean_D_ds = np.mean(D_ds_gold)

    theta_E_gold = gold_truth['main_deflector_parameters_theta_E'].to_numpy()
    mu_theta_E = np.mean(theta_E_gold)

    gamma_gold = gold_truth['main_deflector_parameters_gamma'].to_numpy()
    mu_gamma = np.mean(gamma_gold)
    median_gamma = np.median(gamma_gold)

    z_src_dbls = np.abs(dbls_truth['source_parameters_z_source'].to_numpy())
    z_lens_dbls = np.abs(dbls_truth['lens_light_parameters_z_source'].to_numpy())
    gamma_dbls = dbls_truth['main_deflector_parameters_gamma'].to_numpy()
    dbls_mu_gamma = np.mean(gamma_dbls)

    m_app_ps = gold_truth['point_source_parameters_mag_app'].to_numpy()
    min_pointsource_magapp = np.min(m_app_ps)

    td01 = gold_truth['td01'].to_numpy()
    td02 = gold_truth['td02'].to_numpy()
    td03 = gold_truth['td03'].to_numpy()
    longest_td = np.abs(td03)
    longest_td[np.isnan(longest_td)] = np.abs(td01[np.isnan(longest_td)])
    mean_longest_td = np.mean(longest_td)
    mean_longest_td_prec = np.mean(100 * 5/longest_td)
    max_longest_td = np.max(longest_td)

    fpd01 = gold_truth['fpd01'].to_numpy()
    fpd02 = gold_truth['fpd02'].to_numpy()
    fpd03 = gold_truth['fpd03'].to_numpy()
    longest_fpd = np.abs(fpd03)
    longest_fpd[np.isnan(longest_fpd)] = np.abs(fpd01[np.isnan(longest_fpd)])
    mean_longest_fpd = np.mean(longest_fpd)

    beta_ani = gold_truth['beta_ani'].to_numpy()
    lambda_int = gold_truth['lambda_int'].to_numpy()
    mean_beta_ani = np.mean(beta_ani)
    mean_lambda_int = np.mean(lambda_int)

    test_chain = curr_chain[:,burnin:,:].reshape((-1,curr_chain.shape[2]))
    #sigma_h0 = np.std(test_chain[:,0],ddof=1)
    arviz_hdi = az.hdi(test_chain[:,0], hdi_prob=.68)
    sigma_h0 = (arviz_hdi[1] - arviz_hdi[0])/2
    arviz_hdi = az.hdi(test_chain[:,4], hdi_prob=.68)
    sigma_mu_lint = (arviz_hdi[1] - arviz_hdi[0])/2
    arviz_hdi = az.hdi(test_chain[:,1], hdi_prob=.68)
    sigma_omegaM = (arviz_hdi[1] - arviz_hdi[0])/2
    print('seed %d'%(j))
    print('sigma h0: ', sigma_h0)
    print('sigma omegaM: ', sigma_omegaM)
    zp,de_fom = DE_fom(curr_chain,burnin)
    de_fom_list.append(de_fom)


    chosen_param1 = mu_z_lens
    chosen_param2 = mu_z_src
    plt.scatter(chosen_param1,chosen_param2,label='seed %d'%(j),s=180,c=de_fom,cmap='viridis',norm=defom_norm)

#print(corr_of_param)

plt.xlabel('mean_z_lens',fontsize=15)
plt.ylabel('mean_z_src',fontsize=15)
plt.colorbar()
#plt.legend(fontsize=12)
plt.title('IFU Portion, Pantheon+ $\Omega_m$ prior')
#plt.ylim([2.,22.])

In [ ]:
lens1199 = truth_df[truth_df['catalog_idx'].isin([1199])].reset_index(drop=True)
for key in lens1199:
    print(key, lens1199[key][0])

In [ ]:
import arviz as az
burnin=20000
chains = [
    np.transpose(chains_dict['exp0_2']['w0wa_seed1']['chain'][:,:50000], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed2']['chain'][:,:50000], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed3']['chain'][:,:50000], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed4']['chain'][:,:50000], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed5']['chain'][:,:50000], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed6']['chain'][:,:50000], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed7']['chain'][:,:50000], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed8']['chain'][:,:50000], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed9']['chain'][:,:50000], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed10']['chain'][:,:50000], axes=(1,0,2))
]

de_fom_list = []
plt.figure(dpi=200)
for j,curr_chain in enumerate(chains):
    test_chain = curr_chain[:,burnin:,:].reshape((-1,curr_chain.shape[2]))
    sigma_h0 = np.std(test_chain[:,0],ddof=1)
    #sigma_OmegaM = np.std(test_chain[:,1],ddof=1)
    arviz_hdi = az.hdi(test_chain[:,1], hdi_prob=.68)
    sigma_OmegaM = (arviz_hdi[1] - arviz_hdi[0])/2
    sigma_beta_ani = np.std(test_chain[:,6],ddof=1)
    zp,de_fom = DE_fom(curr_chain,burnin)
    de_fom_list.append(de_fom)
    if j == 5:
        plt.scatter(sigma_OmegaM,de_fom,label='seed %d'%(j),s=180,marker='*',edgecolors='peru')
    else:
        plt.scatter(sigma_OmegaM,de_fom,label='seed %d'%(j),s=90)

plt.ylabel('DE FOM [$\sigma(w_0) \sigma(w_p)$]$^{-1}$',fontsize=15)
plt.xlabel('$\sigma(\Omega_M)$',fontsize=17)
#plt.legend(fontsize=12)
#plt.xlim([0.07,0.094])
#plt.ylim([2.5,13])
plt.title('Flat $\Omega_M$ prior')
plt.savefig('/Users/smericks/Desktop/baseline_fluctuation_OmegaM_0Mprior.pdf')

In [ ]:
import matplotlib

exp_chains = [
    np.transpose(chains_dict['exp0_2']['w0wa_seed7']['chain'],axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed4']['chain'],axes=(1,0,2))]
exp_names = [
             'Exp 0.2: Seed 6',
             'Exp 0.2: Seed 3']

num_chains = len(exp_chains)
burnin = [30000]*num_chains
cmap = plt.get_cmap('ocean')
colors = ['C6','C3']
truth_colors = ["#000000"] * num_chains

custom_lines = []
custom_labels = []

for i,exp_chain in enumerate(exp_chains):

    num_params = exp_chain.shape[2]

    my_color = colors[i]
    
    print(exp_names[i])
    #median_and_uncertainty(exp_chain,burnin[i])
    print(exp_names[i])
    zp,fom = DE_fom(exp_chain,burnin[i])
     
    if i ==0:

        figure = corner.corner(exp_chain[:,burnin[i]:,:-2].reshape((-1,exp_chain.shape[2]-2)),plot_datapoints=False,
            color=my_color,levels=[0.68,0.95],fill_contours=True,
            labels= ['$H_0$','$\Omega_M$','$w_0$','$w_a$',
                r'$\mu(\lambda_{int})$',r'$\sigma(\lambda_{int})$',
                r'$\mu(\beta_{ani})$',r'$\sigma(\beta_{ani})$'],
            dpi=300,truths=[70.,0.3,-1.0,0.,1.,0.1,0.,0.1],truth_color=truth_colors[i],
            fig=None,label_kwargs={'fontsize':24},
            smooth=3)

    else:

        corner.corner(exp_chain[:,burnin[i]:,:-2].reshape((-1,exp_chain.shape[2]-2)),plot_datapoints=False,
            color=my_color,levels=[0.68,0.95],fill_contours=True,
            labels=['$H_0$','$\Omega_M$','$w_0$','$w_a$',
                r'$\mu(\lambda_{int})$',r'$\sigma(\lambda_{int})$',
                r'$\mu(\beta_{ani})$',r'$\sigma(\beta_{ani})$'],
            dpi=300,truths=[70.,0.3,-1.0,0.,1.,0.1,0.,0.1],truth_color=truth_colors[i],
            fig=figure,label_kwargs={'fontsize':24},smooth=3)
        
    custom_lines.append(Line2D([0], [0], color=my_color, lw=4))

    # calculate h0 constraint
    h0, h0_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,0].reshape((-1,1)),weights=None)
    OmegaM, OmegaM_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,1].reshape((-1,1)),weights=None)
    w0, w0_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,2].reshape((-1,1)),weights=None)
    wa, wa_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,3].reshape((-1,1)),weights=None)
    # construct label
    custom_labels.append(exp_names[i]+
        ':\n $H_0$=%.2f$\pm$%.2f \n $\Omega_M$=%.2f$\pm$%.2f \n DE FOM = %.2f'%(
        h0, h0_sigma, OmegaM, OmegaM_sigma, fom))

"""
axes = np.array(figure.axes).reshape((3, 3))
bounds = [[63,77],[1.91,2.095],[0.0,0.2]]
for r in range(0,3):
        for c in range(0,r+1):
            if bounds is not None:
                axes[r,c].set_xlim(bounds[c])
                if r != c :
                    axes[r,c].set_ylim(bounds[r])

axes = np.array(figure.axes).reshape((3, 3))
"""

axes = np.array(figure.axes).reshape((8, 8))
axes[0,7].legend(custom_lines,custom_labels,frameon=False,fontsize=30)
plt.savefig('/Users/smericks/Desktop/lens_selection_comp.pdf')

In [ ]:
truth_df = pd.read_csv('DataVectors/gold/truth_metadata.csv')
for key in truth_df.keys():
    print(key)


In [ ]:
ten_seeds = [
    np.transpose(chains_dict['exp0_2']['w0wa_seed1']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed2']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed3']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed4']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed5']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed6']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed7']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed8']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed9']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed10']['chain'], axes=(1,0,2))
]

fig,axs = plt.subplots(2,2,dpi=200,figsize=(10,10))

burnin = 20000
truth_vals = [70.,0.3,-1.,0.,1.,0.1,0.,0.1]
sigma_baseline = []
for i,chain in enumerate(ten_seeds):

    samps = chain[:,burnin:,:].reshape((-1,exp_chain.shape[2]))
    samps = samps[:,:8]
    print('samps shape: ', samps.shape)
    med = np.median(samps,axis=0)
    low = np.quantile(samps,q=0.1586,axis=0)
    high = np.quantile(samps,q=0.8413,axis=0)

    error = med - truth_vals
    sigma = ((high-med)+(med-low))/2
    if i == 0:
        sigma_baseline = sigma
    bias = error/sigma

    axs[0,0].scatter(0,(sigma[0]),s=30,label='Seed %d'%(i))
    axs[0,1].scatter(0,(sigma[1]),s=30,label='Seed %d'%(i))
    axs[1,0].scatter(0,(sigma[2]),s=30,label='Seed %d'%(i))
    axs[1,1].scatter(0,(sigma[3]),s=30,label='Seed %d'%(i))

    #plt.plot([0,1,2,3,4,5,6,7], (sigma)/truth_vals * 100,label='Seed %d'%(i))
    #plt.scatter([0,1,2,3,4,5,6,7], (sigma)/truth_vals * 100, s=20)

#plt.hlines(y=0.,xmin=0.,xmax=7.,color='black')
axs[0,0].set_title('$\sigma_{H_0}$')
axs[0,1].set_title('$\sigma_{\Omega_M}$')
axs[1,0].set_title('$\sigma_{w_0}$')
axs[1,1].set_title('$\sigma_{w_a}$')
#plt.xticks(ticks=[0,1,2,3,4,5,6,7],labels=['$H_0$','$\Omega_M$','$w_0$','$w_a$',
#    r'$\mu(\lambda_{int})$',r'$\sigma(\lambda_{int})$',
#    r'$\mu(\beta_{ani})$',r'$\sigma(\beta_{ani})$'])
#plt.axhspan(-1, 1, color='grey', alpha=0.2)
axs[0,0].legend(loc='lower left',fontsize=10)

In [ ]:
np.median(de_fom_list)

In [ ]:
np.mean(de_fom_list)

In [ ]:
de_fom_list

In [ ]:
for key in truth_df.keys():
    print(key)

In [ ]:
from scipy.stats import norm
two_seeds = [
    np.transpose(chains_dict['exp0_2']['w0wa_seed1']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed2']['chain'], axes=(1,0,2))
]
color_seeds =['C0','C1']
seed_nums = [0,1]
marker_seeds = ['X','P']

truth_df = pd.read_csv('DataVectors/gold/truth_metadata.csv')

all_z_src = truth_df['source_parameters_z_source'].to_numpy()
all_z_lens = truth_df['lens_light_parameters_z_source'].to_numpy()
all_longest_td = np.abs(truth_df['td03'].to_numpy())
all_longest_td[np.isnan(all_longest_td)] = np.abs(truth_df['td01'].to_numpy()[np.isnan(all_longest_td)])
all_lens_mag = truth_df['lens_light_parameters_mag_app'].to_numpy()

fig,axs = plt.subplots(2,2,dpi=200,figsize=(9,9))

burnin = 20000
truth_vals = [70.,0.3,-1.,0.,1.,0.1,0.,0.1]
sigma_baseline = []
for i,chain in enumerate(two_seeds):

    cidx = np.load('InferenceRuns/exp0_2/catalog_idxs_seed%d.npy'%(seed_nums[i]))

    cidx_ifu = cidx[:50]

    # Index into truth_df by catalog_idx
    # gold_df = gold_df[~gold_df['catalog_idx'].isin(nirspec_quads_catalog_idxs)].reset_index(drop=True)
    jwst_truth = truth_df[truth_df['catalog_idx'].isin(cidx_ifu[:10])].reset_index(drop=True)
    ifu_truth = truth_df[truth_df['catalog_idx'].isin(cidx_ifu)].reset_index(drop=True)

    z_src_ifu = ifu_truth['source_parameters_z_source'].to_numpy()
    z_src_jwst = jwst_truth['source_parameters_z_source'].to_numpy()
    z_lens_ifu = ifu_truth['lens_light_parameters_z_source'].to_numpy()
    z_lens_jwst = jwst_truth['lens_light_parameters_z_source'].to_numpy()
    td01 = ifu_truth['td01'].to_numpy()
    td02 = ifu_truth['td02'].to_numpy()
    td03 = ifu_truth['td03'].to_numpy()
    fpd01 = ifu_truth['fpd01'].to_numpy()
    fpd02 = ifu_truth['fpd02'].to_numpy()
    fpd03 = ifu_truth['fpd03'].to_numpy()
    td01_jwst = jwst_truth['td01'].to_numpy()
    td02_jwst = jwst_truth['td02'].to_numpy()
    td03_jwst = jwst_truth['td03'].to_numpy()
    theta_E_ifu = ifu_truth['main_deflector_parameters_theta_E'].to_numpy()
    theta_E_jwst = jwst_truth['main_deflector_parameters_theta_E'].to_numpy()
    lambda_int_ifu = ifu_truth['lambda_int'].to_numpy()
    lambda_int_jwst = jwst_truth['lambda_int'].to_numpy()
    beta_ani_ifu = ifu_truth['beta_ani'].to_numpy()
    beta_ani_jwst = jwst_truth['beta_ani'].to_numpy()
    gamma_truth_ifu = ifu_truth['main_deflector_parameters_gamma'].to_numpy()
    
    gamma_sigma_jwst = []
    gamma_sigma_ifu = np.empty(np.shape(td01))
    fpd03_sigma_jwst = []
    fpd03_sigma_ifu = np.empty(np.shape(td01))
    ifu_cidx = ifu_truth['catalog_idx'].to_numpy()
    jwst_cidx = jwst_truth['catalog_idx'].to_numpy()
    h5_posteriors_file = 'DataVectors/gold/quad_posteriors_DEBIASED.h5'
    with h5py.File(h5_posteriors_file, 'r') as h5:
        lens_param_samps = h5['lens_param_samps'][:]
        fpd_samps = h5['fpd_samps'][:]
        for cidx in jwst_cidx:
            idx = np.where(h5['catalog_idxs'][:] == cidx)[0]
            print('idx: ', idx)
            gamma_samps = lens_param_samps[idx,:,3]
            gamma_sigma_jwst.append(np.std(gamma_samps,ddof=1))
            fpd03_sigma_jwst.append(np.std(fpd_samps[idx,:,2],ddof=1))

        for k,cidx in enumerate(ifu_cidx):
            if not np.isnan(td03[k]):
                idx = np.where(h5['catalog_idxs'][:] == cidx)[0]
                gamma_samps = lens_param_samps[idx,:,3]
                gamma_sigma_ifu[k] = np.std(gamma_samps,ddof=1)
                fpd03_sigma_ifu[k] = np.std(fpd_samps[idx,:,2],ddof=1)

    h5_posteriors_file = 'DataVectors/gold/dbl_posteriors_DEBIASED.h5'
    with h5py.File(h5_posteriors_file, 'r') as h5:
        lens_param_samps = h5['lens_param_samps'][:]
        fpd_samps = h5['fpd_samps'][:]

        for k,cidx in enumerate(ifu_cidx):
            if np.isnan(td03[k]):
                idx = np.where(h5['catalog_idxs'][:] == cidx)[0]
                fpd03_sigma_ifu[k] = np.std(fpd_samps[idx,:,0],ddof=1)
                gamma_samps = lens_param_samps[idx,:,3]
                gamma_sigma_ifu[k] = np.std(gamma_samps,ddof=1)

    axs[0,0].scatter(z_lens_ifu, z_src_ifu - z_lens_ifu,s=25, 
        marker=marker_seeds[i],label='Seed %d'%(seed_nums[i]),color=color_seeds[i])
    
    longest_td = np.abs(td03)
    longest_td[np.isnan(longest_td)] = np.abs(td01[np.isnan(longest_td)])
    longest_fpd = np.abs(fpd03)
    longest_fpd[np.isnan(longest_fpd)] = np.abs(fpd01[np.isnan(longest_fpd)])

    axs[0,1].scatter(z_src_ifu, longest_td,s=25,
        marker=marker_seeds[i],label='Seed %d'%(seed_nums[i]),color=color_seeds[i],zorder=200)
    
    axs[1,0].scatter(z_src_ifu - z_lens_ifu, theta_E_ifu,s=25,
        marker=marker_seeds[i],label='Seed %d'%(seed_nums[i]),color=color_seeds[i])
    
    # beta_ani prior vs actual
    bani_range = np.arange(-0.23,0.23,0.001)
    axs[1,1].plot(bani_range,
        norm.pdf(bani_range,loc=np.mean(beta_ani_ifu),scale=np.std(beta_ani_ifu,ddof=1)),
        label='Seed %d: $\mu$=%.2f'%(seed_nums[i],np.mean(beta_ani_ifu)),color=color_seeds[i])


axs[0,1].scatter(all_z_src,all_longest_td,s=10,alpha=0.5,color='lightgrey',label='Whole Pop.',edgecolors='none')

axs[1,1].plot(bani_range,
    norm.pdf(bani_range,loc=0.,scale=0.1),
    label='Whole Pop.',color='grey') 
    

axs[0,0].set_xlabel('$z_{lens}$')
axs[0,0].set_ylabel('$z_{src} - z_{lens}$')
axs[0,0].legend(fontsize=8)

axs[0,1].set_xlabel('$z_{src}$')
axs[0,1].set_ylabel('longest $\Delta t$')
axs[0,1].legend(fontsize=8)

axs[1,0].set_xlabel('$z_{src} - z_{lens}$')
axs[1,0].set_ylabel(r'$\theta_E$')
axs[1,0].legend(fontsize=8)

axs[1,1].set_xlabel(r'$\beta_{ani}$')
axs[1,1].legend(fontsize=8)
    #plt.plot([0,1,2,3,4,5,6,7], (sigma)/truth_vals * 100,label='Seed %d'%(i))
    #plt.scatter([0,1,2,3,4,5,6,7], (sigma)/truth_vals * 100, s=20)

In [ ]:
from scipy.stats import norm
from astropy.cosmology import FlatLambdaCDM
gt_cosmo = FlatLambdaCDM(H0=70.,Om0=0.3)

two_seeds = [
    np.transpose(chains_dict['exp0_2']['w0wa_seed1']['chain'], axes=(1,0,2)),
    np.transpose(chains_dict['exp0_2']['w0wa_seed10']['chain'], axes=(1,0,2))
]
color_seeds =['C0','C9']
seed_nums = [0,9]
marker_seeds = ['X','P']

truth_df = pd.read_csv('DataVectors/gold/truth_metadata.csv')

all_z_src = truth_df['source_parameters_z_source'].to_numpy()
all_z_lens = truth_df['lens_light_parameters_z_source'].to_numpy()
all_D_ds = np.array(gt_cosmo.angular_diameter_distance_z1z2(all_z_lens,all_z_src))
all_longest_td = np.abs(truth_df['td03'].to_numpy())
all_longest_td[np.isnan(all_longest_td)] = np.abs(truth_df['td01'].to_numpy()[np.isnan(all_longest_td)])
all_theta_E = truth_df['main_deflector_parameters_theta_E'].to_numpy()
all_lens_mag = truth_df['lens_light_parameters_mag_app'].to_numpy()
all_fpd01 = truth_df['fpd01'].to_numpy()
all_fpd02 = truth_df['fpd02'].to_numpy()
all_fpd03 = truth_df['fpd03'].to_numpy()


fpd03_sigma_all = np.empty(np.shape(all_z_src))
all_cidx = truth_df['catalog_idx'].to_numpy()
h5_posteriors_file = 'DataVectors/gold/quad_posteriors_DEBIASED.h5'
with h5py.File(h5_posteriors_file, 'r') as h5:
    fpd_samps = h5['fpd_samps'][:]

    for k,cidx in enumerate(all_cidx):
        if not np.isnan(all_fpd03[k]):
            idx = np.where(h5['catalog_idxs'][:] == cidx)[0]
            fpd03_sigma_all[k] = np.std(fpd_samps[idx,:,2],ddof=1)

h5_posteriors_file = 'DataVectors/gold/dbl_posteriors_DEBIASED.h5'
with h5py.File(h5_posteriors_file, 'r') as h5:
    fpd_samps = h5['fpd_samps'][:]

    for k,cidx in enumerate(all_cidx):
        if np.isnan(all_fpd03[k]):
            idx = np.where(h5['catalog_idxs'][:] == cidx)[0]
            fpd03_sigma_all[k] = np.std(fpd_samps[idx,:,0],ddof=1)

plt.figure()
plt.scatter(all_z_lens,all_z_src,c=all_D_ds)
plt.xlabel('z_lens')
plt.ylabel('z_src')
plt.title('D_ds (Mpc)')
plt.colorbar()

burnin = 20000
truth_vals = [70.,0.3,-1.,0.,1.,0.1,0.,0.1]
sigma_baseline = []
for i,chain in enumerate(two_seeds):

    cidx = np.load('InferenceRuns/exp0_2/catalog_idxs_seed%d.npy'%(seed_nums[i]))

    cidx_gold = cidx[:50]

    # Index into truth_df by catalog_idx
    # gold_df = gold_df[~gold_df['catalog_idx'].isin(nirspec_quads_catalog_idxs)].reset_index(drop=True)
    gold_truth = truth_df[truth_df['catalog_idx'].isin(cidx_gold)].reset_index(drop=True)

    z_src_gold = gold_truth['source_parameters_z_source'].to_numpy()
    z_lens_gold = gold_truth['lens_light_parameters_z_source'].to_numpy()
    D_ds = np.array(gt_cosmo.angular_diameter_distance_z1z2(z_lens_gold, z_src_gold))
    theta_E_gold = gold_truth['main_deflector_parameters_theta_E'].to_numpy()
    td01 = gold_truth['td01'].to_numpy()
    td02 = gold_truth['td02'].to_numpy()
    td03 = gold_truth['td03'].to_numpy()
    fpd01 = gold_truth['fpd01'].to_numpy()
    fpd02 = gold_truth['fpd02'].to_numpy()
    fpd03 = gold_truth['fpd03'].to_numpy()

    longest_td = np.abs(td03)
    longest_td[np.isnan(longest_td)] = np.abs(td01[np.isnan(longest_td)])

    largest_fpd = np.abs(fpd03)
    largest_fpd[np.isnan(largest_fpd)] = np.abs(fpd01[np.isnan(largest_fpd)])

    fpd03_sigma = np.empty(np.shape(td01))
    gold_cidx = gold_truth['catalog_idx'].to_numpy()
    h5_posteriors_file = 'DataVectors/gold/quad_posteriors_DEBIASED.h5'
    with h5py.File(h5_posteriors_file, 'r') as h5:
        fpd_samps = h5['fpd_samps'][:]

        for k,cidx in enumerate(gold_cidx):
            if not np.isnan(td03[k]):
                idx = np.where(h5['catalog_idxs'][:] == cidx)[0]
                fpd03_sigma[k] = np.std(fpd_samps[idx,:,2],ddof=1)

    h5_posteriors_file = 'DataVectors/gold/dbl_posteriors_DEBIASED.h5'
    with h5py.File(h5_posteriors_file, 'r') as h5:
        fpd_samps = h5['fpd_samps'][:]

        for k,cidx in enumerate(gold_cidx):
            if np.isnan(td03[k]):
                idx = np.where(h5['catalog_idxs'][:] == cidx)[0]
                fpd03_sigma[k] = np.std(fpd_samps[idx,:,0],ddof=1)

    my_samps = np.vstack((D_ds,z_lens_gold,z_src_gold - z_lens_gold,longest_td,largest_fpd)).T
    my_color = color_seeds[i]

    if i ==0:

        figure = corner.corner(my_samps,plot_datapoints=False,plot_density=False,
            color=my_color,levels=[0.68,0.95],fill_contours=False,#contourf_kwargs={"alpha": 0.4},
            labels= ['D_ds','z_lens','z_src - z_lens','longest td','longest_fpd'],
            dpi=300,fig=None,label_kwargs={'fontsize':15},smooth=1.3,
            hist_kwargs={'density':True})

    else:

        corner.corner(my_samps,plot_datapoints=False,plot_density=False,
            color=my_color,levels=[0.68,0.95],fill_contours=False,#contourf_kwargs={"alpha": 0.4},
            labels=['D_ds','z_lens','z_src - z_lens','longest td','longest_fpd'],
            dpi=300,fig=figure,label_kwargs={'fontsize':15},smooth=1.3,hist_kwargs={'density':True})
        
    axes = np.array(figure.axes).reshape((my_samps.shape[1], my_samps.shape[1]))
    for j in range(my_samps.shape[1]):
        for k in range(j):
            ax = axes[j, k]
            ax.scatter(my_samps[:,k],my_samps[:,j],color=my_color,zorder=200)

all_largest_fpd = np.abs(all_fpd03)
all_largest_fpd[np.isnan(all_largest_fpd)] = np.abs(all_fpd01[np.isnan(all_largest_fpd)])
axes[4,1].scatter(all_z_lens,all_largest_fpd,s=30,color='grey',zorder=100)

print(all_theta_E[np.where((all_largest_fpd > 2.) & (all_lens_mag < 22))[0]])
print(all_fpd03[np.where((all_largest_fpd > 2.) & (all_z_lens < 0.5) & (all_lens_mag < 22))[0]])

In [ ]:
fig,axs = plt.subplots(4,1,figsize=(6,13),dpi=200)
exp1_4_idx = np.where((all_theta_E > 1.2) & np.isnan(truth_df['td03'].to_numpy()))[0] 

_,bins,_ = axs[0].hist(all_longest_td,histtype='step',density=True,label='All Lenses',
    color='darkgrey',linewidth=1.5)
axs[0].hist(all_longest_td[exp1_4_idx],histtype='step',density='True',
    label='Exp 1.4',color='deepskyblue',bins=bins,linewidth=1.5)
#plt.legend(fontsize=15)
axs[0].set_title('longest $\Delta t$')
axs[0].legend(fontsize=10)
#plt.title('longest $\Delta t$',fontsize=17)

_,bins,_ = axs[1].hist(all_largest_fpd,histtype='step',density=True,label='All Lenses',
    color='darkgrey',linewidth=1.5)
axs[1].hist(all_largest_fpd[exp1_4_idx],histtype='step',density='True',
    label='Exp 1.4',color='deepskyblue',bins=bins,linewidth=1.5)
axs[1].set_title('largest $\Delta \phi$')

_,bins,_ = axs[2].hist(fpd03_sigma_all,histtype='step',density=True,label='All Lenses',
    color='darkgrey',linewidth=1.5)
axs[2].hist(fpd03_sigma_all[exp1_4_idx],histtype='step',density='True',
    label='Exp 1.4',color='deepskyblue',bins=bins,linewidth=1.5)
axs[2].set_title('largest $\sigma(\Delta \phi)$')

_,bins,_ = axs[3].hist(100*fpd03_sigma_all/all_largest_fpd,histtype='step',density=True,
    label='All Lenses',color='darkgrey',linewidth=1.5)
axs[3].hist(100*fpd03_sigma_all[exp1_4_idx]/all_largest_fpd[exp1_4_idx],histtype='step',density='True',
    label='Exp 1.4',color='deepskyblue',bins=bins,linewidth=1.5)
axs[3].set_title('$\%$ constraint on $\Delta \phi$')

plt.savefig('/Users/smericks/Desktop/exp1_4_hists.pdf')

exp_chains = [
    np.transpose(chains_dict['exp1_2']['w0wa_seed1']['chain'],axes=(1,0,2)),
    np.transpose(chains_dict['exp1_4']['w0wa_seed1']['chain'],axes=(1,0,2))]
exp_names = [
             'Exp 1.2: Quads Favored + $\Delta t$ > 30 days',
             r'Exp 1.4: Doubles Only + $\theta_E$ > 1.2']

num_chains = len(exp_chains)
burnin = [20000]*num_chains
colors = ['goldenrod','deepskyblue']
truth_colors = ["#000000"] * num_chains

custom_lines = []
custom_labels = []

for i,exp_chain in enumerate(exp_chains):

    num_params = exp_chain.shape[2]

    my_color = colors[i]
    
    print(exp_names[i])
    #median_and_uncertainty(exp_chain,burnin[i])
    zp,fom = DE_fom(exp_chain,20000)
     
    if i ==0:

        figure = corner.corner(exp_chain[:,burnin[i]:,:-6].reshape((-1,exp_chain.shape[2]-6)),plot_datapoints=False,
            color=my_color,levels=[0.68,0.95],fill_contours=True,
            labels= ['$H_0$','$\Omega_M$','$w_0$','$w_a$',
                r'$\mu(\lambda_{int})$',r'$\sigma(\lambda_{int})$',
                r'$\mu(\beta_{ani})$',r'$\sigma(\beta_{ani})$'],
            dpi=300,truths=[70.,0.3,-1.0,0.],truth_color=truth_colors[i],
            fig=None,label_kwargs={'fontsize':24},
            smooth=1.5)

    else:

        corner.corner(exp_chain[:,burnin[i]:,:-6].reshape((-1,exp_chain.shape[2]-6)),plot_datapoints=False,
            color=my_color,levels=[0.68,0.95],fill_contours=True,
            labels=['$H_0$','$\Omega_M$','$w_0$','$w_a$',
                r'$\mu(\lambda_{int})$',r'$\sigma(\lambda_{int})$',
                r'$\mu(\beta_{ani})$',r'$\sigma(\beta_{ani})$'],
            dpi=300,truths=[70.,0.3,-1.0,0.],truth_color=truth_colors[i],
            fig=figure,label_kwargs={'fontsize':24},smooth=1.5)
        
    custom_lines.append(Line2D([0], [0], color=my_color, lw=4))

    # calculate h0 constraint
    h0, h0_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,0].reshape((-1,1)),weights=None)
    OmegaM, OmegaM_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,1].reshape((-1,1)),weights=None)
    w0, w0_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,2].reshape((-1,1)),weights=None)
    wa, wa_sigma = median_sigma_from_samples(exp_chain[:,burnin[i]:,3].reshape((-1,1)),weights=None)
    # construct label
    custom_labels.append(exp_names[i]+
        ':\n $H_0$=%.2f$\pm$%.2f \n $\Omega_M$=%.2f$\pm$%.2f \n DE FOM = %.2f'%(
        h0, h0_sigma, OmegaM, OmegaM_sigma, fom))

"""
axes = np.array(figure.axes).reshape((3, 3))
bounds = [[63,77],[1.91,2.095],[0.0,0.2]]
for r in range(0,3):
        for c in range(0,r+1):
            if bounds is not None:
                axes[r,c].set_xlim(bounds[c])
                if r != c :
                    axes[r,c].set_ylim(bounds[r])

axes = np.array(figure.axes).reshape((3, 3))
"""

axes = np.array(figure.axes).reshape((4, 4))
axes[0,3].legend(custom_lines,custom_labels,frameon=False,fontsize=12)
plt.savefig('/Users/smericks/Desktop/exp1_4_contour.pdf')

In [ ]:
len(np.where((all_z_src > 2.2) & (all_longest_td > 200.) & (all_lens_mag < 24.))[0])

In [ ]:
gold_df = pd.read_csv('DataVectors/gold/truth_metadata.csv')

num_muse=40
# TODO: MUSE first, more stringent cut...
longest_td = np.abs(gold_df['td03'].to_numpy()) # 3rd td for quads
longest_td[np.isnan(longest_td)] = np.abs(gold_df['td01'].to_numpy())[np.isnan(longest_td)] # 1st td for doubles
muse_lenses_avail = np.where(
    (longest_td > 200.) & 
    (gold_df['source_parameters_z_source'].to_numpy() > 2.2) & 
    (gold_df['lens_light_parameters_mag_app'].to_numpy() < 22.) 
)
catalog_idx_avail = gold_df.loc[muse_lenses_avail,'catalog_idx'].to_numpy()
print(catalog_idx_avail)
muse_catalog_idxs = np.random.choice(catalog_idx_avail,
    size=num_muse,replace=False)

muse_df = gold_df[gold_df['catalog_idx'].isin(muse_catalog_idxs)].reset_index(drop=True)

muse_quads_catalog_idxs = muse_df[~np.isnan(muse_df['td03'].to_numpy())]['catalog_idx'].to_numpy()
muse_dbls_catalog_idxs = muse_df[np.isnan(muse_df['td03'].to_numpy())]['catalog_idx'].to_numpy()

In [ ]:
plt.hist(all_z_lens,density=True,histtype='step')


highz_idx = np.where((all_z_src > 2.5))[0]
plt.hist(all_z_lens[highz_idx],density=True,histtype='step')

nice_idx = np.where((all_z_src > 2.5) & (all_longest_td > 200.))[0]
plt.hist(all_z_lens[nice_idx],density=True,histtype='step')


### TODO: Check if beta_ani / J relation is Gaussian at all ###

In [ ]:
LENS_IDX = 2

from DataVectors.prep_data_vectors import gaussianize_samples

posteriors_h5_file = 'DataVectors/gold/quad_posteriors_KIN.h5'
kinematic_type = 'MUSE'
num_gaussianized_samps = 5000
# load in from posteriors file
with h5py.File(posteriors_h5_file, "r") as h5:

    # set-up indexing
    h5_catalog_idxs = h5['catalog_idxs'][:]
    my_idxs = np.arange(0,len(h5_catalog_idxs))

    fpd_samps = h5['fpd_samps'][my_idxs]
    lens_param_samps = h5['lens_param_samps'][my_idxs]
    beta_ani_samps = h5['beta_ani_samps'][my_idxs]
    h5_catalog_idxs = h5['catalog_idxs'][my_idxs]

    # pull c_sqrtJ_samps based on kinematic type
    if kinematic_type is not None:
        if kinematic_type == '4MOST':
            c_sqrtJ_samps = h5['c_sqrtJ_samps'][my_idxs]
        elif kinematic_type == 'MUSE':
            c_sqrtJ_samps = h5['MUSE_c_sqrtJ_samps'][my_idxs]
        elif kinematic_type == 'NIRSPEC':
            c_sqrtJ_samps = h5['NIRSPEC_c_sqrtJ_samps'][my_idxs]
        else:
            raise ValueError("kinematic_type not supported")
        
        num_kin_bins = np.shape(c_sqrtJ_samps)[-1]

num_lenses = np.shape(fpd_samps)[0]
num_td = np.shape(fpd_samps)[-1]
if num_gaussianized_samps is not None:
    to_gaussianize_input = []
    # fpds
    for i in range(0,num_td):
        to_gaussianize_input.append(fpd_samps[:,:,i])
    # gamma_lens
    to_gaussianize_input.append(lens_param_samps[:,:,3])
    gamma_idx = num_td
    if kinematic_type is not None:
        # beta_ani
        to_gaussianize_input.append(beta_ani_samps)
        beta_idx = num_td+1
        # sigma_v bins
        for j in range(0,num_kin_bins):
            to_gaussianize_input.append(c_sqrtJ_samps[:,:,j]**2)

    to_gaussianize_input = np.asarray(to_gaussianize_input)
    # switch 1st dim to last dim (parameters dim)
    input_samps = np.transpose(to_gaussianize_input,axes=(1,2,0))
    # now gaussianize
    gaussian_samps = np.empty((num_lenses,
        num_gaussianized_samps,np.shape(input_samps)[-1]))
    for l_idx in range(0,num_lenses):
        gaussian_samps[l_idx] = gaussianize_samples(
            input_samps[l_idx],num_gaussianized_samps)

# deal with edge cases of 1 td, 1 kinematic bin
# 1 td
gaussian_fpd_samps = gaussian_samps[:,:,0:num_td]
#if num_td == 1:
#    gaussian_fpd_samps = gaussian_fpd_samps[:,:,np.newaxis]
# 1 kin bin
if kinematic_type is not None:
    gaussian_kin_samps = gaussian_samps[:,:,-num_kin_bins:]


figure = corner.corner(input_samps[LENS_IDX],plot_datapoints=False,
    color='indianred',levels=[0.68,0.95],fill_contours=True,
    labels= ['$\Delta \phi_{01}$','$\Delta \phi_{02}$',
        '$\Delta \phi_{03}$','$\gamma_{lens}$', r'$\beta_{ani}$',
        '$c^2 \mathcal{J} Bin1$ (km/s)','$c^2 \mathcal{J} Bin2$ (km/s)',
        '$c^2 \mathcal{J} Bin3$ (km/s)'],
    dpi=300,fig=None,label_kwargs={'fontsize':24},smooth=0.7,hist_kwargs={'density':True})

figure = corner.corner(gaussian_samps[LENS_IDX],plot_datapoints=False,
    color='cornflowerblue',levels=[0.68,0.95],fill_contours=True,
    labels= ['$\Delta \phi_{01}$','$\Delta \phi_{02}$',
        '$\Delta \phi_{03}$','$\gamma_{lens}$', r'$\beta_{ani}$',
        '$c^2 \mathcal{J} Bin1$ (km/s)','$c^2 \mathcal{J} Bin2$ (km/s)',
        '$c^2 \mathcal{J} Bin3$ (km/s)'],
    dpi=300,fig=figure,label_kwargs={'fontsize':24},smooth=0.7,hist_kwargs={'density':True})



custom_lines = [Line2D([0], [0], color='indianred', lw=4),Line2D([0], [0], color='cornflowerblue', lw=4)]
custom_labels = ['500 Input Samples', 'Gaussianized Samples']

"""
axes = np.array(figure.axes).reshape((3, 3))
bounds = [[63,77],[1.91,2.095],[0.0,0.2]]
for r in range(0,3):
        for c in range(0,r+1):
            if bounds is not None:
                axes[r,c].set_xlim(bounds[c])
                if r != c :
                    axes[r,c].set_ylim(bounds[r])

axes = np.array(figure.axes).reshape((3, 3))
"""
n_params = np.shape(gaussian_samps)[-1]
axes = np.array(figure.axes).reshape((n_params, n_params))
axes[0,n_params-1].legend(custom_lines,custom_labels,frameon=False,fontsize=25)